In [ ]:
import os,sys,re,math,time,json,requests
from datetime import datetime
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from termcolor import colored, cprint
import asyncio,random
from concurrent.futures import ThreadPoolExecutor

def default_string():
    return ''
from transcript_map import video_transcript_dict, question_id_map_slide_dict, slide_summary_dict

import openai
from openai import OpenAI

# ضع مفتاح OpenAI الخاص بك هنا بين علامات التنصيص
my_api_key = "YOUR_OPENAI_API_KEY"

openai.api_key = my_api_key
client = OpenAI(api_key=my_api_key)

# ================================================================================
# ===== 🔑 إضافة مفاتيح API ودوال استدعاء النماذج المتعددة =====
# ================================================================================

# مفتاح Hugging Face للنماذج العربية
HUGGINGFACE_API_KEY = "YOUR_HUGGINGFACE_TOKEN"

from huggingface_hub import InferenceClient
hf_client = InferenceClient(token=HUGGINGFACE_API_KEY)

# قائمة النماذج (بدون Claude)
MODEL_NAMES = {
    2: "gpt-3.5-turbo",
    3: "gpt-4o-mini",
    4: "llama-3.1-8b",
    5: "qwen2.5-7b",
    6: "silma-9b"
}

MODEL_DISPLAY = {
    2: "GPT-3.5",
    3: "GPT-4o Mini",
    4: "Llama 3.1",
    5: "Qwen2.5 7B (عربي)",
    6: "Phi-3 Medium"  # تم استبدال SILMA بـ Phi-3
}

# أسماء نماذج Hugging Face الصحيحة
HF_MODEL_NAMES = {
    "llama-3.1-8b": "meta-llama/Llama-3.1-8B-Instruct",
    "qwen2.5-7b": "Qwen/Qwen2.5-7B-Instruct",
    "silma-9b": "microsoft/Phi-3-medium-4k-instruct"  # استبدال SILMA بـ Phi-3 (متاح ومجاني)
}

# دالة استدعاء OpenAI مع إعادة المحاولة
def call_openai(messages, model, max_tokens=2048, temperature=0.7, retry_count=0):
    import time
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        if "rate_limit" in str(e) or "429" in str(e):
            # استخراج وقت الانتظار من رسالة الخطأ
            wait_time = 0.5
            if "Please try again in" in str(e):
                import re
                match = re.search(r'(\d+)ms', str(e))
                if match:
                    wait_time = int(match.group(1)) / 1000 + 0.1
                else:
                    match = re.search(r'(\d+\.?\d*)s', str(e))
                    if match:
                        wait_time = float(match.group(1)) + 0.1

            backoff = wait_time * (1.2 ** min(retry_count, 10))
            backoff = min(backoff, 60)

            print(f"⏸️ حد API - انتظار {backoff:.1f}s (محاولة {retry_count+1})")
            time.sleep(backoff)
            return call_openai(messages, model, max_tokens, temperature, retry_count + 1)

        print(f"❌ خطأ OpenAI: {e}")
        if retry_count < 3:
            print(f"🔄 إعادة المحاولة بعد 2 ثانية...")
            time.sleep(2)
            return call_openai(messages, model, max_tokens, temperature, retry_count + 1)
        return ""

# دالة استدعاء Hugging Face
def call_huggingface(messages, model_name, max_tokens=2048, temperature=0.7):
    import time
    for i in range(3):
        try:
            response = hf_client.chat_completion(
                model=model_name,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"⚠️ محاولة {i+1}/3 لـ {model_name}: {e}")
            time.sleep(2 ** i)
    return ""

# دالة موحدة لاستدعاء أي نموذج
def call_model(messages, model_type):
    """استدعاء موحد لكل النماذج"""
    if model_type == 2:
        return call_openai(messages, "gpt-3.5-turbo")
    elif model_type == 3:
        return call_openai(messages, "gpt-4o-mini")
    elif model_type == 4:
        return call_huggingface(messages, HF_MODEL_NAMES["llama-3.1-8b"])
    elif model_type == 5:
        return call_huggingface(messages, HF_MODEL_NAMES["qwen2.5-7b"])
    elif model_type == 6:
        return call_huggingface(messages, HF_MODEL_NAMES["silma-9b"])
    else:
        return ""

# متغير النموذج العام (سيتم تغييره في الحلقة)
SELECTED_MODEL = 3  # القيمة الافتراضية: GPT-4o Mini

print("✅ تم إعداد دوال النماذج المتعددة (5 نماذج)")

# ================================================================================

class Avatar(object):
    # agent id should be the same as the real student's ID
    # each agent's persona, memory, and reflection will be stored into json file. The prompt/response history will be stored into a separate text file.
    def __init__(self,agent_config,agent_id):
        self.agent_config = agent_config
        self.agent_id = agent_id
        self.persona = ''

        self.sim_config_name = ['reflection_choice','memory_component_choice','memory_source','forget_effect','sim_strategy','example_demo','gpt_type']
        self.sim_config_set = [self.agent_config[sim_item] for sim_item in self.sim_config_name]
        self.memory_component_choice_list = self.agent_config['memory_component_choice'].split('+')
        self.choice_map = {'أ':0,'ب':1,'ج':2,'د':3}
        self.choice_map_en = {'A':0,'B':1,'C':2,'D':3}  # للنموذج الذي يرجع حروف إنجليزية

        self._check_result_file()
        self._make_assertion()
        self._make_result_folder()
        self._load_dataset()
        self._init_agent_dataset()
        self._init_result_write()

    def _check_result_file(self):
        if os.path.exists(self.agent_config['result_path']+'/result_ind_dur.csv'):
            with open(self.agent_config['result_path']+'/result_ind_dur.csv', 'r') as file:
                for i,line in enumerate(file):
                    if len(line.split(',')) != 40:
                        print(i,line)
                        assert 1 == 0
        if os.path.exists(self.agent_config['result_path']+'/result_ind_post.csv'):
            with open(self.agent_config['result_path']+'/result_ind_post.csv', 'r') as file:
                for i,line in enumerate(file):
                    if len(line.split(',')) != 11:
                        print(i,line)
                        assert 1 == 0


    def _make_assertion(self):
        assert self.agent_config['memory_source'] in ['real','sim']
        assert self.agent_config['forget_effect'] in ['no_memory','random_half_plus_recent_one','all_plus_recent_one','only_recent_one']
        assert self.agent_config['reflection_choice'] in ['yes','no']
        assert self.agent_config['memory_component_choice'] in ['KM','KM+PM','KM+PM+MM','KM+PM+MM+CM','KM+PM+CM','KM+MM+CM']
        assert self.agent_config['sim_strategy'] in ['standard', 'cot', 'react', 'standard_cog', 'cot_cog', 'react_cog']
        assert self.agent_config['example_demo'] in ['yes','no']
        # assert self.agent_config['gpt_type'] in [0,1,2,3,4]


    def _make_result_folder(self):


        self.root_folder = self.agent_config['result_path'] + '/' + self.agent_config['memory_source'] + '_' + self.agent_config['forget_effect'] + '_reflect-' + self.agent_config['reflection_choice'] + '_' + self.agent_config['memory_component_choice'] + '_' + self.agent_config['sim_strategy'] + '_example-' + self.agent_config['example_demo'] + '_' + str(self.agent_config['gpt_type'])

        # التأكد من إنشاء المجلد الأساسي والمجلدات الفرعية لتجنب الأخطاء
        if not os.path.exists(self.root_folder):
            os.makedirs(self.root_folder)
        if not os.path.exists(self.root_folder + '/log/'):
            os.makedirs(self.root_folder + '/log/')
        if not os.path.exists(self.root_folder + '/agent_memory/'):
            os.makedirs(self.root_folder + '/agent_memory/')
        if not os.path.exists(self.root_folder + '/user_memory/'):
            os.makedirs(self.root_folder + '/user_memory/')

        self.log_file = self.root_folder + '/log/' + str(self.agent_id) + '.txt'
        self.agent_memory_file = self.root_folder + '/agent_memory/' + str(self.agent_id) + '.json'
        self.user_memory_file = self.root_folder + '/user_memory/' + str(self.agent_id) + '.json'

        # self.sim_result_dur_path = self.agent_config['result_path']+'/result_ind_dur.csv'
        # self.sim_result_post_path = self.agent_config['result_path']+'/result_ind_post.csv'

        self.sim_result_dur_path = self.root_folder+'/log/'+str(self.agent_id)+'_result_ind_dur.csv'
        self.sim_result_post_path = self.root_folder+'/log/'+str(self.agent_id)+'_result_ind_post.csv'

        self._store_log('\n\n Start running new simulation program '+ '='*100 +'\n\n')

    def _load_dataset(self):
            # 1. ملفات البيانات الديموغرافية (تستخدم ;)
            self.demo_dataset_origin = pd.read_csv(self.agent_config['dataset_path']+'/student_demo.csv', encoding='utf-8-sig', sep=';')
            self.demo_dataset_extend = pd.read_csv(self.agent_config['dataset_path']+'/student_demo_generated.csv', encoding='utf-8-sig', sep=';')

            # 2. انتبه هنا: هذا الملف يستخدم الفاصلة العادية (,) حسب الفحص السابق
            self.course_dataset = pd.read_csv(self.agent_config['dataset_path']+'/course_material_slide.csv', encoding='utf-8-sig', sep=',')

            # 3. باقي الملفات تستخدم (;)
            self.student_answer_item_dataset = pd.read_csv(self.agent_config['dataset_path']+'/student_answer_item_revised.csv', encoding='utf-8-sig', sep=';')
            self.during_dataset = pd.read_csv(self.agent_config['dataset_path']+'/during_behavior_slide.csv', encoding='utf-8-sig', sep=',')
            self.aoi_material_dataset = pd.read_csv(self.agent_config['dataset_path']+'/aoi_material_ext_slide.csv', encoding='utf-8-sig', sep=';')
            self.question_dataset = pd.read_csv(self.agent_config['dataset_path']+'/student_question.csv', encoding='utf-8-sig', sep=';')

            self.agent_id_to_course_dict = self._get_agent_id_to_course_dict()
            self.course_to_transcript_dict = self._get_course_to_transcript_dict()

    def _init_agent_dataset(self):
        self.video_id = self.agent_id_to_course_dict[self.agent_id]
        self.example_id_list = self.agent_config['example_user_dict']['video_'+str(self.video_id)][0:1]

        self.transcript_id_list_all = self.course_to_transcript_dict[self.video_id]
        self.transcript_id_list_all.sort()
        self.transcript_id_list_simulation = self.transcript_id_list_all
        self.transcript_num_all = len(self.transcript_id_list_all)

        self.question_dataset_agent = self.question_dataset[self.question_dataset['course_name']==self.video_id]
        self.question_id_list = list(set(self.question_dataset_agent['question_id']))
        self.question_id_list.sort()

        self.transcript_dict_agent = video_transcript_dict['video_'+str(self.video_id)]
        self.course_dataset_agent = self.course_dataset[self.course_dataset['course_name']==self.video_id]
        self.aoi_material_dataset_agent = self.aoi_material_dataset[self.aoi_material_dataset['course_name']==self.video_id]

        self.student_answer_item_dataset_agent = self.student_answer_item_dataset[self.student_answer_item_dataset['student_id']==self.agent_id]
        self.during_dataset_agent = self.during_dataset[self.during_dataset['student_id']==self.agent_id]
        self.demo_agent = self.demo_dataset_origin[self.demo_dataset_origin['student_id']==self.agent_id]
        if len(self.demo_agent) == 0:
            self.demo_agent = self.demo_dataset_extend[self.demo_dataset_extend['student_id']==self.agent_id]
            if len(self.demo_agent) == 0:
                print('Agent ID can not be found...')
                assert 1==0

    def _get_result_specific_config(self,result_table):
        result_table_new = result_table.copy()
        for config_name in self.sim_config_name:
            result_table_new = result_table_new[result_table_new[config_name]==self.agent_config[config_name]]
        result_table_new = result_table_new[result_table_new['student_id']==self.agent_id]
        return result_table_new

    def _init_result_write(self):
        self.metric_ind_dur_list = ['gaze_aoi_accuracy','gaze_aoi_distance','motor_aoi_accuracy','motor_aoi_distance','workload_diff','curiosity_diff','valid_focus_diff','follow_ratio_diff','engagement_accuracy','confusion_accuracy']
        self.metric_ind_question_list = ['choice_similarity','accuracy_similarity']

        dur_result_list = ['gaze_aoi_id','gaze_aoi_center_tuple_x','gaze_aoi_center_tuple_y','motor_aoi_id','motor_aoi_center_tuple_x','motor_aoi_center_tuple_y','workload','curiosity','valid_focus','course_follow','engagement','confusion']
        user_dur_result_list = ['user_'+d_item for d_item in dur_result_list]
        agent_dur_result_list = ['agent_'+d_item for d_item in dur_result_list]

        if os.path.exists(self.sim_result_dur_path):
            # حذفنا sep=';' لأن الملف يتم حفظه بفاصلة عادية
            self.exist_simulation_result_dur = pd.read_csv(self.sim_result_dur_path, encoding='utf-8-sig')
            self.exist_simulation_result_dur_specific_config = self._get_result_specific_config(self.exist_simulation_result_dur)
            self.exist_simulation_transcript_id_list = list(set(self.exist_simulation_result_dur_specific_config['transcript_id']))
            self.exist_simulation_transcript_id_list.sort()
        else:
            self.exist_simulation_result_dur = None
            self.exist_simulation_transcript_id_list = []
            with open(self.sim_result_dur_path,'a+', encoding='utf-8-sig') as fwrite:
                fwrite.write(','.join(self.sim_config_name+['student_id','transcript_id','sentence_id']+user_dur_result_list+agent_dur_result_list+self.metric_ind_dur_list)+'\n')

        if os.path.exists(self.sim_result_post_path):
            # حذفنا sep=';' لأن الملف يتم حفظه بفاصلة عادية
            self.exist_simulation_result_post = pd.read_csv(self.sim_result_post_path, encoding='utf-8-sig')
            self.exist_simulation_result_post_specific_config = self._get_result_specific_config(self.exist_simulation_result_post)
            self.exist_simulation_question_id_list = list(set(self.exist_simulation_result_post_specific_config['question_id']))
        else:
            self.exist_simulation_result_post = None
            self.exist_simulation_question_id_list = []
            with open(self.sim_result_post_path,'a+', encoding='utf-8-sig') as fwrite:
                fwrite.write(','.join(self.sim_config_name+['student_id','question_id']+['user_answer','agent_answer','correct_answer']+self.metric_ind_question_list)+'\n')

    def _get_agent_id_to_course_dict(self):
        student_id_list_origin = list(set(self.demo_dataset_origin['student_id']))
        student_id_list_extend = list(set(self.demo_dataset_extend['student_id']))
        student_id_list = student_id_list_origin + student_id_list_extend
        agent_id_to_course_dict = {}
        for student_id in student_id_list:
            if student_id in student_id_list_origin:
                demo_data_item = self.demo_dataset_origin[self.demo_dataset_origin['student_id']==student_id]
            elif student_id in student_id_list_extend:
                demo_data_item = self.demo_dataset_extend[self.demo_dataset_extend['student_id']==student_id]
            else:
                assert 1==0
            agent_id_to_course_dict[student_id] = demo_data_item['video_id'].values[0]
        return agent_id_to_course_dict

    def _get_course_to_transcript_dict(self):
        course_id_list = list(set(self.course_dataset['course_name']))
        course_to_transcript_dict = {}
        for course_id in course_id_list:
            course_data_item = self.course_dataset[self.course_dataset['course_name']==course_id]
            course_to_transcript_dict[course_id] = list(set(course_data_item['slide_id_from_zero']))
            course_to_transcript_dict[course_id].sort()
        return course_to_transcript_dict

    def _get_aoi_choice_str(self,aoi_material_table):
        # course_name   start_timestamp end_timestamp   transcript_id   aoi_id  aoi_upper_left_x    aoi_upper_left_y    aoi_lower_right_x   aoi_lower_right_y   aoi_center_x    aoi_center_y    aoi_content
        aoi_id_list = list(set(aoi_material_table['aoi_id']))
        aoi_id_list.sort()
        aoi_num = len(aoi_id_list)
        # تم تعريب النص هنا
        aoi_material_choice_str = f'\n هناك {aoi_num} مناطق اهتمام (AOIs). معرف كل منطقة ومحتواها مدرج أدناه: \n'
        for aoi_id in aoi_id_list:
            aoi_material_item = aoi_material_table[aoi_material_table['aoi_id']==aoi_id]
            aoi_content = aoi_material_item['aoi_content'].values[0]
            aoi_material_choice_str += f'#منطقة الاهتمام {aoi_id}#: {aoi_content}. \n'

        return aoi_material_choice_str

    def _generate_persona(self,demo_item):
        # تم تعريب صياغة الجمل لكي تظهر شخصية الطالب بشكل سليم باللغة العربية
        if 'attitude' in demo_item.columns:
            # video_id,student_id,age,gender,major,education,attitude,exam,focus,curiosity,interest,priors,compliance,smartness,family
            persona = ('الطالب رقم #' + str(demo_item['student_id'].values[0]) + ' جنسه ' + str(demo_item['gender'].values[0]) + ' وعمره ' + str(demo_item['age'].values[0]) + ' . المستوى التعليمي لهذا الطالب هو ' +
                  str(demo_item['education'].values[0]) + ' في تخصص ' + str(demo_item['major'].values[0]) + '. علاوة على ذلك، هذا الطالب ' +
                  str(demo_item['attitude'].values[0]) + ' في الدورة و ' + str(demo_item['exam'].values[0]) + '. أثناء الدورة، هذا الطالب ' +
                  str(demo_item['focus'].values[0]) + ' و هو ' + str(demo_item['curiosity'].values[0]) + '. بالإضافة إلى ذلك، هذا الطالب ' +
                  str(demo_item['interest'].values[0]) + ' ولديه ' + str(demo_item['priors'].values[0]) + '. أيضاً، هذا الطالب ' +
                  str(demo_item['smartness'].values[0]) + ' و ' + str(demo_item['compliance'].values[0]) + '. أخيراً، خلفية عائلة هذا الطالب ' +
                  str(demo_item['family'].values[0]) + '. \n'
                  )
        else:
            persona = ('الطالب رقم #' + str(demo_item['student_id'].values[0]) + ' وهو ' + str(demo_item['gender'].values[0]) + ' يبلغ من العمر ' + str(demo_item['age'].values[0]) + ' عاماً، تعليمه ' +
                  str(demo_item['education'].values[0]) + ' في تخصص ' + str(demo_item['major'].values[0]) + '. مستوى إلمام الطالب بتعلم الآلة هو ' +
                  str(demo_item['ML_familarity'].values[0]) + '، ومستوى الخبرة في استخدام تقنيات الذكاء الاصطناعي لحل المشكلات هو ' + str(demo_item['ML_Experience'].values[0]) +
                  ' ومعدل خبرة تعلم الآلة الإجمالي هو ' + str(demo_item['ML_bg_rate'].values[0]) + '(1: لا توجد خبرة، 5: خبير). \n')
        return persona

    def instantiate_profile(self):
        # gender,age,major,education,ML_familarity,ML_Experience,ML_bg_rate
        self.persona = self._generate_persona(self.demo_agent)
        self._store_log('Finish instantiating persona: '+self.persona+'\n\n')

    def instantiate_memory(self):
        # التعامل مع ملف ذاكرة المستخدم (User Memory)
        if os.path.exists(self.user_memory_file):
            try:
                # تمت إضافة encoding='utf-8' للقراءة الصحيحة
                with open(self.user_memory_file, 'r', encoding='utf-8') as json_file:
                    self.agent_real_memory_stream = json.load(json_file)
            except:
                self.agent_real_memory_stream = []
                # تمت إضافة encoding='utf-8' و ensure_ascii=False للكتابة الصحيحة
                with open(self.user_memory_file, 'w', encoding='utf-8') as json_file:
                    json.dump(self.agent_real_memory_stream, json_file, indent=4, ensure_ascii=False)
        else:
            self.agent_real_memory_stream = []
            with open(self.user_memory_file, 'w', encoding='utf-8') as json_file:
                json.dump(self.agent_real_memory_stream, json_file, indent=4, ensure_ascii=False)

        # التعامل مع ملف ذاكرة الوكيل (Agent Memory)
        if os.path.exists(self.agent_memory_file):
            try:
                with open(self.agent_memory_file, 'r', encoding='utf-8') as json_file:
                    self.agent_sim_memory_stream = json.load(json_file)
            except:
                self.agent_sim_memory_stream = []
                with open(self.agent_memory_file, 'w', encoding='utf-8') as json_file:
                    json.dump(self.agent_sim_memory_stream, json_file, indent=4, ensure_ascii=False)
        else:
            self.agent_sim_memory_stream = []
            with open(self.agent_memory_file, 'w', encoding='utf-8') as json_file:
                json.dump(self.agent_sim_memory_stream, json_file, indent=4, ensure_ascii=False)

        self._store_log('\n\n Finish instantiating memory: '+'\n\n')

    def _get_example_demo_str_per(self,example_id,slide_id,question_id_list):
        # output: demo user persona, gaze AOI ID, motor AOI ID, cognitive states, question choice
        demo_item_example = self.demo_dataset_origin[self.demo_dataset_origin['student_id']==example_id]
        example_persona = self._generate_persona(demo_item_example)

        during_dataset_example = self.during_dataset[self.during_dataset['student_id']==example_id]
        during_item = during_dataset_example[during_dataset_example['slide_id_from_zero']==slide_id]

        sentence_id_list = list(set(during_item['transcript_id']))
        sentence_id_list.sort()
        user_gaze_aoi_id,user_gaze_aoi_center_tuple,user_motor_aoi_id,user_motor_aoi_center_tuple = {},{},{},{}
        user_workload, user_curiosity, user_valid_focus, user_course_follow, user_engagement, user_confusion = {},{},{},{},{},{}
        user_workload_str, user_curiosity_str, user_valid_focus_str, user_course_follow_str, user_engagement_str, user_confusion_str = '', '', '', '', '', ''
        user_gaze_aoi_id_str, user_motor_aoi_id_str = '', ''
        user_dur_counter = {}
        for sentence_id in sentence_id_list:
            during_item_per = during_item[during_item['transcript_id']==sentence_id]
            user_gaze_aoi_id[sentence_id], user_gaze_aoi_center_tuple[sentence_id] = self._get_real_gaze(during_item_per)
            user_motor_aoi_id[sentence_id], user_motor_aoi_center_tuple[sentence_id] = self._get_real_motor(during_item_per)
            user_workload[sentence_id], user_curiosity[sentence_id], user_valid_focus[sentence_id], user_course_follow[sentence_id], user_engagement[sentence_id], user_confusion[sentence_id] = self._get_real_cognitive_state(during_item_per)

            # تم تعريب النصوص التالية لتناسب النموذج العربي
            if user_workload[sentence_id] != None:
                if user_workload_str == '':
                    user_workload_str += f'\n # مسار عبء العمل (Workload) # هو: {user_workload[sentence_id]}, '
                else:
                    user_workload_str += f'{user_workload[sentence_id]}, '
            if user_curiosity[sentence_id] != None:
                if user_curiosity_str == '':
                    user_curiosity_str += f'\n # مسار الفضول (Curiosity) # هو: {user_curiosity[sentence_id]}, '
                else:
                    user_curiosity_str += f'{user_curiosity[sentence_id]}, '
            if user_valid_focus[sentence_id] != None:
                if user_valid_focus_str == '':
                    user_valid_focus_str += f'\n # مسار التركيز الفعلي (Valid Focus) # هو: {user_valid_focus[sentence_id]}, '
                else:
                    user_valid_focus_str += f'{user_valid_focus[sentence_id]}, '
            if user_course_follow[sentence_id] != None:
                if user_course_follow_str == '':
                    user_course_follow_str += f'\n # مسار متابعة الدرس (Course Follow) # هو: {user_course_follow[sentence_id]}, '
                else:
                    user_course_follow_str += f'{user_course_follow[sentence_id]}, '
            if user_engagement[sentence_id] != None:
                if user_engagement_str == '':
                    user_engagement_str += f'\n # مسار التفاعل (Engagement) # هو: {user_engagement[sentence_id]}, '
                else:
                    user_engagement_str += f'{user_engagement[sentence_id]}, '
            if user_confusion[sentence_id] != None:
                if user_confusion_str == '':
                    user_confusion_str += f'\n # مسار الارتباك (Confusion) # هو: {user_confusion[sentence_id]}, '
                else:
                    user_confusion_str += f'{user_confusion[sentence_id]}, '

            if user_gaze_aoi_id[sentence_id] != None:
                if user_gaze_aoi_id_str == '':
                    user_gaze_aoi_id_str += f'\n # مسار منطقة اهتمام النظر (Gaze Watch AOI) # هو: {user_gaze_aoi_id[sentence_id]}, '
                else:
                    user_gaze_aoi_id_str += f'{user_gaze_aoi_id[sentence_id]}, '

            if user_motor_aoi_id[sentence_id] != None:
                if user_motor_aoi_id_str == '':
                    user_motor_aoi_id_str += f'\n # مسار حركة الماوس (Mouse Move AOI) # هو: {user_motor_aoi_id[sentence_id]}, '
                else:
                    user_motor_aoi_id_str += f'{user_motor_aoi_id[sentence_id]}, '

        student_answer_item = self.student_answer_item_dataset[self.student_answer_item_dataset['student_id']==example_id]

        user_answer_letter = {}
        user_answer_letter_str = ''
        question_id_list.sort()
        for question_id in question_id_list:
            user_answer_item = student_answer_item[student_answer_item['question_id']=='test_q'+str(question_id)]
            user_answer_letter[question_id] = user_answer_item['choice'].values[0]

            if user_answer_letter[question_id] != None:
                if user_answer_letter_str == '':
                    # تعريب نص السؤال والاختيار
                    user_answer_letter_str += f'\n # خيار السؤال #: معرف السؤال: {question_id}, اختيار الطالب: {user_answer_letter[question_id]}, '
                else:
                    user_answer_letter_str += f'معرف السؤال: {question_id}, اختيار الطالب: {user_answer_letter[question_id]}, '

        demo_str_per = f'\n\n # مثال الطالب {example_id} #\n'
        demo_str_per += example_persona
        demo_str_per += f'\n في الشريحة الحالية # رقم {slide_id} #، بالنسبة للطالب {example_id}: '

        demo_str_per = demo_str_per + user_workload_str + user_curiosity_str + user_valid_focus_str + user_course_follow_str + user_engagement_str + user_confusion_str + user_gaze_aoi_id_str + user_motor_aoi_id_str + user_answer_letter_str

        return demo_str_per

    def obtain_example_demo_str(self,slide_id,question_id_list):
        if len(self.example_id_list) == 0: return ''
        # تعريب المقدمة والملاحظة الختامية
        demo_str_all = f'\n إليك # أمثلة # لأداء طالب آخر في الشريحة الحالية #{slide_id}#. \n'
        for e_i,example_id in enumerate(self.example_id_list):
            demo_str_per = self._get_example_demo_str_per(example_id,slide_id,question_id_list)
            demo_str_all += demo_str_per
        demo_str_all += '\n لاحظ أن هذه الأمثلة قد تمنحك رؤى ومرجعاً حول سلووكيات تعلم الطلاب، ولكن لا يجب عليك نسخ نتائجهم مباشرة.\n'
        return demo_str_all

    def _find_match_cognitive_gaze_motor(self,resyntax,input_string):
        pattern = re.compile(resyntax, re.IGNORECASE)
        matches = pattern.findall(input_string)
        if matches:
            match_result_list = [{'sentence_id': float(match[0]), 'workload': float(match[1]), 'curiosity': float(match[2]), 'valid_focus': float(match[3]), 'course_follow': float(match[4]), 'engagement': float(match[5]), 'confusion': float(match[6]), 'gaze_aoi_id': float(match[7]), 'motor_aoi_id': float(match[8])} for match in matches]
            return match_result_list
        else:
            return []


    def _find_match_cognitive(self,resyntax,input_string):
        pattern = re.compile(resyntax, re.IGNORECASE)
        matches = pattern.findall(input_string)
        if matches:
            match_result_list = [{'sentence_id': float(match[0]), 'workload': float(match[1]), 'curiosity': float(match[2]), 'valid_focus': float(match[3]), 'course_follow': float(match[4]), 'engagement': float(match[5]), 'confusion': float(match[6])} for match in matches]
            return match_result_list
        else:
            return []

    def _find_match_gaze(self,resyntax,input_string):
        pattern = re.compile(resyntax, re.IGNORECASE)
        matches = pattern.findall(input_string)
        if matches:
            match_result_list = [{'sentence_id': float(match[0]), 'gaze_aoi_id': float(match[1])} for match in matches]
            return match_result_list
        else:
            return []

    def _find_match_motor(self,resyntax,input_string):
        pattern = re.compile(resyntax, re.IGNORECASE)
        matches = pattern.findall(input_string)
        if matches:
            match_result_list = [{'sentence_id': float(match[0]), 'motor_aoi_id': float(match[1])} for match in matches]
            return match_result_list
        else:
            return []

    def _find_match_choice(self,resyntax,input_string):
        pattern = re.compile(resyntax, re.IGNORECASE)
        matches = pattern.findall(input_string)
        if matches:
            match_result_list = [{'question_id': float(match[0]), 'choice': match[1].upper()} for match in matches]
            return match_result_list
        else:
            return []

    def _extract_match_gaze(self,match_gaze_list,aoi_material_table):
        if len(match_gaze_list) != 0:
            gaze_aoi_id,gaze_center_tuple = {},{}
            for match_gaze in match_gaze_list:
                sentence_id_value = match_gaze['sentence_id']
                aoi_id_value = match_gaze['gaze_aoi_id']
                gaze_aoi_id[sentence_id_value] = aoi_id_value
                aoi_piece = aoi_material_table[aoi_material_table['aoi_id']==aoi_id_value]
                if len(aoi_piece) == 0:
                    gaze_aoi_id[sentence_id_value] = None
                    gaze_center_tuple[sentence_id_value] = (None,None)
                else:
                    gaze_aoi_center_x, gaze_aoi_center_y = aoi_piece['aoi_center_x'].values[0], aoi_piece['aoi_center_y'].values[0]
                    gaze_center_tuple[sentence_id_value] = (gaze_aoi_center_x, gaze_aoi_center_y)

            gaze_aoi_id = dict(sorted(gaze_aoi_id.items()))
            gaze_center_tuple = dict(sorted(gaze_center_tuple.items()))
        else:
            gaze_aoi_id,gaze_center_tuple = {}, {}

        return gaze_aoi_id,gaze_center_tuple

    def _extract_match_motor(self,match_mouse_list,aoi_material_table):
        if len(match_mouse_list) != 0:
            move_aoi_id,move_center_tuple = {},{}
            for match_mouse in match_mouse_list:
                sentence_id_value = match_mouse['sentence_id']
                aoi_id_value = match_mouse['motor_aoi_id']
                move_aoi_id[sentence_id_value] = aoi_id_value
                aoi_piece = aoi_material_table[aoi_material_table['aoi_id']==aoi_id_value]
                if len(aoi_piece) == 0:
                    move_aoi_id[sentence_id_value] = None
                    move_center_tuple[sentence_id_value] = (None,None)
                else:
                    move_aoi_center_x, move_aoi_center_y = aoi_piece['aoi_center_x'].values[0], aoi_piece['aoi_center_y'].values[0]
                    move_center_tuple[sentence_id_value] = (move_aoi_center_x, move_aoi_center_y)
            move_aoi_id = dict(sorted(move_aoi_id.items()))
            move_center_tuple = dict(sorted(move_center_tuple.items()))

        else:
            move_aoi_id,move_center_tuple = {}, {}

        return move_aoi_id,move_center_tuple

    def _extract_match_choice(self,match_choice_list):
        if len(match_choice_list) != 0:
            agent_choice = {}
            for match_choice in match_choice_list:
                question_id_value = match_choice['question_id']
                choice_value = match_choice['choice']
                try:
                    # جرب الحروف الإنجليزية أولاً (لأن النموذج يرجع A,B,C,D)
                    if choice_value in self.choice_map_en:
                        agent_choice[question_id_value] = self.choice_map_en[choice_value]
                    # إذا لم تنجح، جرب الحروف العربية
                    elif choice_value in self.choice_map:
                        agent_choice[question_id_value] = self.choice_map[choice_value]
                    else:
                        agent_choice[question_id_value] = None
                except:
                    agent_choice[question_id_value] = None
            agent_choice = dict(sorted(agent_choice.items()))
        else:
            agent_choice = {}

        return agent_choice

    def _extract_match_cognitive(self,match_cognitive_list):
        if len(match_cognitive_list) != 0:
            agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion = {},{},{},{},{},{}
            for match_cognitive in match_cognitive_list:
                sentence_id_value = match_cognitive['sentence_id']
                agent_workload[sentence_id_value] = match_cognitive['workload']
                agent_curiosity[sentence_id_value] = match_cognitive['curiosity']
                agent_valid_focus[sentence_id_value] = match_cognitive['valid_focus']
                agent_course_follow[sentence_id_value] = match_cognitive['course_follow']
                agent_engagement[sentence_id_value] = match_cognitive['engagement']
                agent_confusion[sentence_id_value] = match_cognitive['confusion']
            agent_workload = dict(sorted(agent_workload.items()))
            agent_curiosity = dict(sorted(agent_curiosity.items()))
            agent_valid_focus = dict(sorted(agent_valid_focus.items()))
            agent_course_follow = dict(sorted(agent_course_follow.items()))
            agent_engagement = dict(sorted(agent_engagement.items()))
            agent_confusion = dict(sorted(agent_confusion.items()))
        else:
            agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion = {},{},{},{},{},{}

        return agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion

    def _extract_match_cognitive_gaze_motor(self,match_cognitive_gaze_motor_list,aoi_material_table):
        if len(match_cognitive_gaze_motor_list) != 0:
            agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion,gaze_aoi_id,gaze_center_tuple,move_aoi_id,move_center_tuple = {},{},{},{},{},{},{},{},{},{}
            for match_cognitive_gaze_motor in match_cognitive_gaze_motor_list:
                sentence_id_value = match_cognitive_gaze_motor['sentence_id']
                agent_workload[sentence_id_value] = match_cognitive_gaze_motor['workload']
                agent_curiosity[sentence_id_value] = match_cognitive_gaze_motor['curiosity']
                agent_valid_focus[sentence_id_value] = match_cognitive_gaze_motor['valid_focus']
                agent_course_follow[sentence_id_value] = match_cognitive_gaze_motor['course_follow']
                agent_engagement[sentence_id_value] = match_cognitive_gaze_motor['engagement']
                agent_confusion[sentence_id_value] = match_cognitive_gaze_motor['confusion']

                gaze_aoi_id_value = match_cognitive_gaze_motor['gaze_aoi_id']
                gaze_aoi_id[sentence_id_value] = gaze_aoi_id_value
                gaze_aoi_piece = aoi_material_table[aoi_material_table['aoi_id']==gaze_aoi_id_value]
                if len(gaze_aoi_piece) == 0:
                    gaze_aoi_id[sentence_id_value] = None
                    gaze_center_tuple[sentence_id_value] = (None,None)
                else:
                    gaze_aoi_center_x, gaze_aoi_center_y = gaze_aoi_piece['aoi_center_x'].values[0], gaze_aoi_piece['aoi_center_y'].values[0]
                    gaze_center_tuple[sentence_id_value] = (gaze_aoi_center_x, gaze_aoi_center_y)

                motor_aoi_id_value = match_cognitive_gaze_motor['motor_aoi_id']
                move_aoi_id[sentence_id_value] = motor_aoi_id_value
                motor_aoi_piece = aoi_material_table[aoi_material_table['aoi_id']==motor_aoi_id_value]
                if len(motor_aoi_piece) == 0:
                    move_aoi_id[sentence_id_value] = None
                    move_center_tuple[sentence_id_value] = (None,None)
                else:
                    move_aoi_center_x, move_aoi_center_y = motor_aoi_piece['aoi_center_x'].values[0], motor_aoi_piece['aoi_center_y'].values[0]
                    move_center_tuple[sentence_id_value] = (move_aoi_center_x, move_aoi_center_y)
            agent_workload = dict(sorted(agent_workload.items()))
            agent_curiosity = dict(sorted(agent_curiosity.items()))
            agent_valid_focus = dict(sorted(agent_valid_focus.items()))
            agent_course_follow = dict(sorted(agent_course_follow.items()))
            agent_engagement = dict(sorted(agent_engagement.items()))
            agent_confusion = dict(sorted(agent_confusion.items()))
            gaze_aoi_id = dict(sorted(gaze_aoi_id.items()))
            gaze_center_tuple = dict(sorted(gaze_center_tuple.items()))
            move_aoi_id = dict(sorted(move_aoi_id.items()))
            move_center_tuple = dict(sorted(move_center_tuple.items()))
        else:
            agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion,gaze_aoi_id,gaze_center_tuple,move_aoi_id,move_center_tuple = {},{},{},{},{},{},{},{},{},{}

        return agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion,gaze_aoi_id,gaze_center_tuple,move_aoi_id,move_center_tuple


    def action_gaze_mouse_cog_question_concise(self,sentence_id_list,example_demo_str,sim_strategy,retrieved_memory,transcript_id,transcript_material,aoi_material_table,question_content_dict,choice_content_dict,repeat_threshold=3, current_repetition=0):
        self._store_log('\n Start Gaze Motor Cognitive State Question Simulation '+'-'*40+'\n\n')

        # تعريب مقدمة البرومبت
        sys_prompt = 'افترض أنك ' + self.persona + '.\n المعلومات أعلاه هي # شخصيتك #. '

        task_overview = ('\n افترض أنك في دورة تعليمية عبر الإنترنت. '
            +'مهمتك هي محاكاة # الحالات الإدراكية # للطالب (المهمة 1)، # منطقة اهتمام النظر # (المهمة 2)، # منطقة حركة الماوس # (المهمة 3)، و # اختيار إجابة السؤال # (المهمة 4) '
            +'التفاصيل موضحة أدناه. \n')

        task_detail = ('\n الشريحة في الدورة مقسمة إلى عدة مناطق اهتمام (AOIs) ويمكنك إما النظر إلى كل منطقة أو استخدام فأرة الكمبيوتر لاستكشافها.'
                +'\n تحتوي كل شريحة على قائمة من نصوص الشرح (Transcripts) من المعلم.'
                +'\n مهمتك 1 هي محاكاة مسار الحالات الإدراكية الست لهذا الطالب في قائمة النصوص في الشريحة الحالية، وتشمل:'
                +'\n WORKLOAD (عبء العمل), CURIOSITY (الفضول), VALID FOCUS (التركيز الفعلي), COURSE FOLLOW (متابعة الدورة), ENGAGEMENT (التفاعل), CONFUSION (الارتباك).'
                +'\n كل حالة تتراوح من 0 (مستوى منخفض جداً) إلى 1 (مستوى مرتفع جداً) ويجب أن تكون دقيقة لمنزلتين عشريتين على الأقل.'
                +'\n مهمتك 2 هي محاكاة مسار سلوكيات نظر هذا الطالب وتحديد منطقة الاهتمام (AOI) المحددة التي تشاهدها حالياً أثناء كل نص.'
                +'\n مهمتك 3 هي محاكمة مسار سلوك حركة الماوس لهذا الطالب وتحديد منطقة الاهتمام (AOI) التي تمر عليها بالماوس أثناء كل نص.'
                +'\n مهمتك 4 هي محاكمة أداء تعلم هذا الطالب للإجابة على كل سؤال من أسئلة ما بعد الدورة.\n'
                )


        cog_str = ('\n إليك # استراتيجية المحاكاة # لتتعلم من البيانات السابقة. '
            +'\n لمحاكمة الحالات الإدراكية (المهمة 1): أولاً، ضع في اعتبارك مسارات حالاتك الإدراكية السابقة لأن حالاتك الحالية قد تكون مرتبطة بها.'
            +'\n ثانياً، حلل الارتباطات بين مناطق اهتمام الشرائح السابقة ومسارات النظر/الماوس لترى سلوكيات متابعة الدورة، وما إذا كانت حالاتك الذهنية جيدة أم لا.'
            +'\n مثال: المتابعة القوية للدورة ترتبط بحالات إدراكية جيدة (مثل تركيز عالٍ أو عبء عمل منخفض). '
            +'\n لمحاكمة النظر/الماوس (المهمة 2 و 3): أولاً، ضع في اعتبارك مساراتك السابقة لأن السلوك الحالي يرتبط بالسابق.'
            +'\n ثانياً، ضع في اعتبارك حالاتك الإدراكية لتقرر ما إذا كانت محاكاتك يجب أن تتبع محتوى الدورة الحالي أم لا.'
            +'\n مثال: الحالات الجيدة تعني متابعة قوية، لذا يجب أن يكون نظرك متسقاً مع المحتوى الحالي. '
            +'\n أما الحالات السيئة (مثل عبء العمل العالي) قد تجعلك تبقي نظرك في أماكن سابقة أو مناطق عشوائية. '
            +'\n أخيراً، لاحظ أن سلوك النظر لا يجب أن يكون مطابقاً تماماً لسلوك الماوس.'
            +'\n لمحاكمة إجابة السؤال (المهمة 4): حلل ارتباطاتك السابقة لترى مدى متابعتك للدورة، مما يؤثر على دقة إجابتك.'
            +'\n مثال: المتابعة الجيدة والحالات الذهنية الجيدة قد تشير لإجابات صحيحة، والعكس صحيح. '
            +'\n أخيراً، بناءً على كل ما سبق، قدر ما إذا كنت ستجيب بشكل صحيح أم لا. إذا نعم، اختر الإجابة الصحيحة. إذا لا، اختر إجابة خاطئة محتملة. '
            +'\n لاحظ أن محاكاتك يجب ألا تكون ثابتة دائماً، بل تتغير حسب مواد الدورة والتفاعلات المذكورة. '
        )

        simulation_stratey_standard = '\n # استراتيجية المحاكاة # الخاصة بك: أكمل المهام الأربع وفقاً للتعليمات.'
        simulation_stratey_cot = '\n # استراتيجية المحاكاة #: بدلاً من إعطاء الإجابات مباشرة، فكر كيف تقسم كل مهمة لخطوات ثم نفذها.'
        simulation_stratey_react = '\n # استراتيجية المحاكاة #: فكر في ترتيب إنجاز المهام وكيف تؤثر نتائج كل مهمة على الأخرى.'
        simulation_stratey_standard_cog = simulation_stratey_standard + '\n\n' + cog_str
        simulation_stratey_cot_cog = simulation_stratey_cot + '\n\n' + cog_str
        simulation_stratey_react_cog = simulation_stratey_react + '\n\n' + cog_str

        simulation_stratey_dict = {'standard':simulation_stratey_standard,'cot':simulation_stratey_cot,'react':simulation_stratey_react,'standard_cog':simulation_stratey_standard_cog,'cot_cog':simulation_stratey_cot_cog,'react_cog':simulation_stratey_react_cog}

        simulation_stratey = simulation_stratey_dict[sim_strategy]

        warning = '\n الرجاء استخدام # استراتيجية المحاكاة # لإنهاء المهام الأربع.'

        # هنا الجزء الأهم: إجبار النموذج على استخدام الصيغة الإنجليزية للنتائج
        warning += ('\n ملاحظة هامة جداً: 1. إجابتك للمهام 1، 2، و 3 يجب أن تكون مجمعة معاً وباللغة الإنجليزية حصراً في التنسيق التالي بدقة: \n Transcript ID: [value], WORKLOAD: [value], CURIOSITY: [value], VALID FOCUS: [value], COURSE FOLLOW: [value], ENGAGEMENT: [value], CONFUSION: [value], WATCH AOI: [aoi id], MOUSE MOVE AOI: [aoi id].'
                +f'\n يجب أن تعطي قيمة لكل مهمة لكل نص (transcript) من النص {sentence_id_list[0]} إلى النص {sentence_id_list[-1]} في الشريحة.'
                +'\n يجب أن تعطي قيمة لمحاكمة حركة الماوس ولا تتركها فارغة.'
                +'\n 2. إجابتك للمهمة 4 (الأسئلة) يجب أن تكون بالتنسيق التالي بدقة بالإنجليزية: \n Question ID: [id value], Question Choice: [choice]. '
                +'\n تأكد من وجود رمز ID بعد كلمة Question، ويجب أن يكون الاختيار [choice] واحداً فقط من الحروف الإنجليزية: A,B,C,D بدون أي كلمات أخرى.'
                +f'\n يجب أن تعطي إجابة لكل سؤال.\n '
        )

        if self.agent_config['memory_source'] == 'sim':
            warning += (
                '\n محاكاتك للنظر، الحركة، الحالات الإدراكية وإجابة الأسئلة يجب أن تتكيف مع # شخصية الطالب # المحددة.'
                +'\n على سبيل المثال، في مهمة الأسئلة، هدفك ليس الإجابة بشكل صحيح دائماً، بل محاكمة مستوى الطالب. الطلاب الجيدون يجيبون بشكل صحيح، والطلاب الضعفاء قد يخطئون.'
            )

        warning += '\n فقط أعطني المخرجات للمهام الأربع بالتنسيق المطلوب (Format) باللغة الإنجليزية للأرقام والعناوين. لا تكتب أي مقدمات أو شروحات أخرى.'

        observation = (f'\n الآن أنت في دورة عبر الإنترنت وهذا ما يقوله المعلم: # معرف شريحة الدورة الحالي: {transcript_id}, محتويات الشريحة: {transcript_material} #. \n')

        aoi_material_choice_str = self._get_aoi_choice_str(aoi_material_table)

        question_str = f'\n لقد تعلمت المعرفة من # شريحة الدورة الحالية: {transcript_id}#. الآن ستجيب على أسئلة الاختبار التالية لتقييم أدائك:'
        question_id_list = list(question_content_dict.keys())
        question_id_list.sort()
        for question_id in question_id_list:
            question_content = question_content_dict[question_id]
            choice_content = choice_content_dict[question_id]
            # إجبار النموذج على فهم أن الاختيارات هي A, B, C, D
            question_str += f'\n السؤال {question_id}: # {question_content} #. يجب عليك اختيار إجابة واحدة من الخيارات (استخدم الحرف الإنجليزي فقط A,B,C,D): # {choice_content} # \n'

        content_prompt = task_overview + task_detail + simulation_stratey + retrieved_memory + observation + aoi_material_choice_str + question_str + example_demo_str + warning

        # تعديل استدعاء النموذج لاستخدام GPT دائماً
        # قمنا بإلغاء الشروط المعقدة لأننا نعتمد على OpenAI فقط
        messages = [{"role": "system",
                "content": sys_prompt},
                {"role": "user",
                "content": content_prompt}]

        # استدعاء دالة GPT مباشرة
        llm_response = self._response_llm_gpt(messages)

        match_cognitive_gaze_motor_list = self._find_match_cognitive_gaze_motor(r"Transcript ID: \[?(\d+\.\d+|\d+)\]?, WORKLOAD: \[?(\d+\.\d+|\d+)\]?, CURIOSITY: \[?(\d+\.\d+|\d+)\]?, VALID FOCUS: \[?(\d+\.\d+|\d+)\]?, COURSE FOLLOW: \[?(\d+\.\d+|\d+)\]?, ENGAGEMENT: \[?(\d+\.\d+|\d+)\]?, CONFUSION: \[?(\d+\.\d+|\d+)\]?, WATCH AOI: \[?(\d+\.\d+|\d+)\]?, MOUSE MOVE AOI: \[?(\d+\.\d+|\d+)\]?",llm_response)
        match_choice_list = self._find_match_choice(r"Question ID: \[?(\d+\.\d+|\d+)\]?, Question Choice: \[?([a-zA-Z])\]?",llm_response)

        self._store_log('Repetition: '+str(current_repetition)+'-'*10)
        self._store_log('sys_prompt: '+sys_prompt)
        self._store_log('content_prompt: '+content_prompt)
        self._store_log('\n\n llm response: '+llm_response+'\n\n')

        if len(match_cognitive_gaze_motor_list) == 0 and len(match_choice_list) == 0:
            current_repetition += 1
            print(f"Agent {self.agent_id} in {transcript_id} Function action_gaze_mouse_cog_question_concise returned None. Repetition {current_repetition} of {repeat_threshold}.")

            if current_repetition <= 3:
                return self.action_gaze_mouse_cog_question_concise(sentence_id_list,example_demo_str,sim_strategy,retrieved_memory,transcript_id,transcript_material,aoi_material_table,question_content_dict,choice_content_dict,repeat_threshold, current_repetition)
            else:
                print(f"Agent {self.agent_id} in {transcript_id} Function action_gaze_mouse_cog_question_concise Reached repetition threshold ({repeat_threshold}). Exiting.")
                return {},{},{},{},{},{},{},{},{},{},{}
        else:
            agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion,gaze_aoi_id,gaze_center_tuple,move_aoi_id,move_center_tuple = self._extract_match_cognitive_gaze_motor(match_cognitive_gaze_motor_list,aoi_material_table)
            agent_choice = self._extract_match_choice(match_choice_list)

            return gaze_aoi_id, gaze_center_tuple, move_aoi_id, move_center_tuple, agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion, agent_choice

    def reflect_reason(self,memory_stream_str):
        if len(memory_stream_str) == 0 or memory_stream_str == None: return ''

        self._store_log('\n Start reflect_reason '+'-'*40+'\n\n')

        # تعريب مقدمة البرومبت
        sys_prompt = 'افترض أنك ' + self.persona

        # تعريب المهمة (Task)
        task = ('\n سأعطيك سجل تجاربك السابقة في دورة عبر الإنترنت، والتي قد تتضمن أنماطاً مختلفة مثل مواد الدورة، ومناطق اهتمام نظرك (AOI)، ومناطق حركة الماوس لاستكشاف المحتوى على الشرائح، وحالاتك الإدراكية. ' +
            ' مهمتك هي التأمل والتفكير المنطقي بناءً على الارتباطات والعلاقات بين هذه الأنماط المختلفة. ' +
            ' على سبيل المثال، فكر واستنتج لماذا أدت مواد دراسية معينة إلى سلوكيات نظرك، وسلوكيات حركة الماوس، وحالاتك الإدراكية. وكيف يمكن لهذه الأنماط أن تؤثر على بعضها البعض. \n')

        # تعريب التحذير وتحديد عدد الكلمات
        warning = ('\n يجب أن تكتب نتائج تأملك وتفكيرك مباشرة في حدود 50 كلمة عربية. لا تكتب أي معلومات أخرى. \n')
        content_prompt = task + memory_stream_str + warning

        # تبسيط الكود لاستخدام GPT دائماً وحذف النماذج الأخرى
        messages = [{"role": "system",
                "content": sys_prompt},
                {"role": "user",
                "content": content_prompt}]

        llm_response = self._response_llm_gpt(messages)

        self._store_log('sys_prompt: '+sys_prompt)
        self._store_log('content_prompt: '+content_prompt)
        self._store_log('\n\n llm response: '+llm_response)

        return llm_response


    def _select_memory_index(self,memory_stream,current_transcript_id):
        # هذه الدالة لا تحتاج لتعديل لأنها تعتمد على المنطق البرمجي فقط
        # ['no_memory','random_half_plus_recent_one','all_plus_recent_one','only_recent_one']
        if self.agent_config['forget_effect'] == 'no_memory':
            return []
        elif self.agent_config['forget_effect'] == 'all_plus_recent_one':
            return memory_stream
        elif self.agent_config['forget_effect'] == 'random_half_plus_recent_one':
            half_length = len(memory_stream) // 2
            retrieved_memory_stream = random.sample(memory_stream, half_length)
            return retrieved_memory_stream
        elif self.agent_config['forget_effect'] == 'only_recent_one':
            return memory_stream[-1:]
        else:
            assert 1 == 0


    def retrieve_memory(self,memory_stream_list,current_transcript_id):
        # هذه الدالة منطقية بحتة ولا تحتاج لتعريب أو تعديل
        memory_stream_list_sorted = sorted(memory_stream_list, key=lambda x: x['transcript_id'])
        memory_stream_len = len(memory_stream_list_sorted)

        memory_stream_list_filtered = self._select_memory_index(memory_stream_list_sorted,current_transcript_id)
        # memory_stream_list_filtered = memory_stream_list_sorted

        memory_name_dict = {'PM':['gaze_aoi_id'],'MM':['motor_aoi_id'],'CM':['workload','curiosity','valid_focus','course_follow','engagement','confusion']}
        composed_memory_stream = []
        for mi,memory_element in enumerate(memory_stream_list_filtered):
            transcript_id = memory_element['transcript_id']
            transcript = memory_element['observation']
            action_dict = memory_element['action']
            action_name_list = list(action_dict.keys())

            memory_element_retrieve = {'transcript_id':transcript_id}
            if 'reflection' in list(memory_element.keys()) and self.agent_config['reflection_choice'] == 'yes':
                memory_element_retrieve['reflection'] = memory_element['reflection']
            memory_element_retrieve['observation'] = transcript if 'KM' in self.memory_component_choice_list else ''
            memory_element_retrieve['action'] = {}
            for memory_component_name in self.memory_component_choice_list:
                if memory_component_name == 'KM': continue
                for per_metric in memory_name_dict[memory_component_name]:
                    if per_metric not in action_name_list:
                        memory_element_retrieve['action'][per_metric] = None
                    else:
                        memory_element_retrieve['action'][per_metric] = action_dict[per_metric]
            composed_memory_stream.append(memory_element_retrieve)

        return composed_memory_stream

    def summarize_transcripts_llm(self,memory_stream_list):
        if len(memory_stream_list) == 0 or memory_stream_list == None: return ''

        self._store_log('\n Start summarize_transcripts '+'-'*40+'\n\n')

        # تعريب التوجيه
        sys_prompt = 'أنت مساعد ذكي جيد في تلخيص المواد الدراسية.'

        # تعريب المهمة
        task = ('\n سأعطيك قائمة بنصوص الدورة أدناه. مهمتك هي تلخيص هذه النصوص الدراسية في حدود 50 كلمة.\n ')

        transcripts_str = ''
        for memory_element in memory_stream_list:
            transcript_id = memory_element['transcript_id']
            transcript_content = memory_element['observation']
            transcripts_str += f'#Transcript: {transcript_id}#: {transcript_content}. \n'

        # تعريب التحذير
        warning = ('\n يجب أن تكتب ملخص النصوص مباشرة في حدود 50 كلمة. لا تخرج أي معلومات أخرى. \n')
        content_prompt = task + transcripts_str + warning

        # استخدام GPT فقط
        messages = [{"role": "system",
                "content": sys_prompt},
                {"role": "user",
                "content": content_prompt}]
        llm_response = self._response_llm_gpt(messages)

        self._store_log('sys_prompt: '+sys_prompt)
        self._store_log('content_prompt: '+content_prompt)
        self._store_log('\n\n llm response: '+llm_response)

        return llm_response

    def summarize_gaze_llm(self,transcript_list):
        if len(transcript_list) == 0 or transcript_list == None: return ''

        self._store_log('\n Start summarize_gaze '+'-'*40+'\n\n')

        # تعريب التوجيه
        sys_prompt = 'أنت مساعد ذكي جيد في تلخيص المواد الدراسية ورصد سلوكيات نظر الطلاب.'

        # تعريب المهمة
        task = ('\n سأعطيك قائمة بنصوص مناطق الشريحة في الدورة التي شاهدها الطالب بالتسلسل. ' +
                'مهمتك هي تلخيص سلوكيات نظر الطالب على هذه النصوص في حدود 50 كلمة. ')

        transcripts_str = '\n فيما يلي مناطق الدورة التي شاهدها الطالب.\n'
        for t_id,transcript_content in enumerate(transcript_list):
            transcripts_str += f'#نظرة على النص: {t_id}#: {transcript_content}. \n'

        # تعريب التحذير
        warning = ('\n يجب أن تكتب ملخصك مباشرة في حدود 50 كلمة. لا تخرج أي معلومات أخرى. \n')
        content_prompt = task + transcripts_str + warning

        # استخدام GPT فقط
        messages = [{"role": "system",
                "content": sys_prompt},
                {"role": "user",
                "content": content_prompt}]
        llm_response = self._response_llm_gpt(messages)

        self._store_log('sys_prompt: '+sys_prompt)
        self._store_log('content_prompt: '+content_prompt)
        self._store_log('\n\n llm response: '+llm_response)

        return llm_response



    def summarize_motor_llm(self,transcript_list):
        if len(transcript_list) == 0 or transcript_list == None: return ''

        self._store_log('\n Start summarize_motor '+'-'*40+'\n\n')

        # تعريب التوجيه
        sys_prompt = 'أنت مساعد ذكي جيد في تلخيص المواد الدراسية ورصد سلوكيات حركة الماوس للطلاب في الدورة التدريبية عبر الإنترنت.'

        # تعريب المهمة
        task = ('\n سأعطيك قائمة بنصوص مناطق الشرائح في الدورة، التي استكشفها الطالب بالتسلسل باستخدام فأرة الكمبيوتر. ' +
                'مهمتك هي تلخيص سلوكيات استكشاف حركة الماوس للطالب على هذه النصوص في حدود 50 كلمة. ')

        transcripts_str = '\n فيما يلي نصوص الدورة التي استكشفها الطالب.\n'
        for t_id,transcript_content in enumerate(transcript_list):
            transcripts_str += f'#حركة الماوس على النص: {t_id}#: {transcript_content}. \n'

        # تعريب التحذير
        warning = ('\n يجب أن تكتب ملخصك مباشرة في حدود 50 كلمة. لا تخرج أي معلومات أخرى. \n')
        content_prompt = task + transcripts_str + warning

        # استخدام GPT فقط
        messages = [{"role": "system",
                "content": sys_prompt},
                {"role": "user",
                "content": content_prompt}]
        llm_response = self._response_llm_gpt(messages)

        self._store_log('sys_prompt: '+sys_prompt)
        self._store_log('content_prompt: '+content_prompt)
        self._store_log('\n\n llm response: '+llm_response)

        return llm_response

    def summarize_transcripts(self,memory_stream_list):
        if len(memory_stream_list) == 0 or memory_stream_list == None: return ''
        transcripts_str = ''
        for memory_element in memory_stream_list:
            transcript_id = memory_element['transcript_id']
            # تعريب العنوان
            transcripts_str += f'#الشريحة: {transcript_id}#:  \n'
            course_material_item = self.course_dataset_agent[self.course_dataset_agent['slide_id_from_zero']==transcript_id]
            tiny_transcript_id_list = list(set(course_material_item['transcript_id']))
            tiny_transcript_id_list.sort()
            transcripts_str += self._get_transcript_str(tiny_transcript_id_list)
            transcripts_str += '\n'
            # topic_sum = slide_summary_dict['video_'+str(self.video_id)][transcript_id]
            # transcripts_str += f'#Slide: {transcript_id}#: this slide talks about {topic_sum}. \n'
        return transcripts_str

    def summarize_gaze_max(self,gaze_tuple_list):
        if len(gaze_tuple_list) == 0 or gaze_tuple_list == None: return ''
        # تعريب المقدمة
        transcripts_str = '# منطقة اهتمام النظر (Gaze Watch AOI) #: ملخص لمسار نظرك أدناه. \n'

        valid_num = 0
        for d in gaze_tuple_list:
            if d[0] == None or len(d[0]) == 0: continue
            aoi_id_dict,transcript_id = d[0],d[1]
            value_counts = Counter(aoi_id_dict.values())
            aoi_id_frequent = max(value_counts, key=value_counts.get)
            if aoi_id_frequent == None: continue
            aoi_material_match = self.aoi_material_dataset_agent[self.aoi_material_dataset_agent['slide_id_from_zero']==transcript_id]
            aoi_material_match_item = aoi_material_match[aoi_material_match['aoi_id']==aoi_id_frequent]
            aoi_content = aoi_material_match_item['aoi_content'].values[0]
            # تعريب الجملة الوصفية
            transcripts_str += f'أثناء الشريحة: {transcript_id}, ركز نظر الطالب أكثر على المحتوى أدناه: # {aoi_content} #. \n'
            valid_num += 1

        if valid_num == 0: return ''

        return transcripts_str

    def summarize_gaze(self,gaze_tuple_list):
        if len(gaze_tuple_list) == 0 or gaze_tuple_list == None: return ''
        # تعريب المقدمة
        transcripts_str = '# منطقة اهتمام النظر (Gaze Watch AOI) #: ملخص لمسار نظرك أدناه. \n'
        valid_num = 0
        for d in gaze_tuple_list:
            if d[0] in [-1,None] or len(d[0]) == 0: continue
            aoi_id_dict,transcript_id = d[0],d[1]
            # aoi_material_match = self.aoi_material_dataset_agent[self.aoi_material_dataset_agent['slide_id_from_zero']==transcript_id]
            aoi_id_dict_sorted = dict(sorted(aoi_id_dict.items()))
            action_trajectory = list(aoi_id_dict_sorted.values())

            # for aoi_id_each in action_trajectory:
            action_trajectory = [str(a_t) if a_t is not None else ' ' for a_t in action_trajectory]
            action_trajectory = ','.join(action_trajectory)
            # aoi_material_match_item = aoi_material_match[aoi_material_match['aoi_id']==aoaoi_id_eachi_id]
            # aoi_content = aoi_material_match_item['aoi_content'].values[0]
            # gaze_aoi_trajectory =
            # تعريب وصف المسار
            transcripts_str += f'أثناء الشريحة: {transcript_id}, مسار معرفات مناطق النظر (AOI ID trajectory) هو: {action_trajectory}\n'
            valid_num += 1
        if valid_num == 0: return ''

        return transcripts_str



    def summarize_motor_max(self,motor_tuple_list):
        if len(motor_tuple_list) == 0 or motor_tuple_list == None: return ''
        # تعريب المقدمة
        transcripts_str = '# منطقة حركة الماوس (Mouse Move AOI) #: ملخص لمسار حركة الماوس الخاص بك أدناه. \n'
        # print('motor_tuple_list: ',motor_tuple_list)
        valid_num = 0
        for d in motor_tuple_list:
            if d[0] == None or len(d[0]) == 0: continue
            aoi_id_dict,transcript_id = d[0],d[1]
            value_counts = Counter(aoi_id_dict.values())
            aoi_id_frequent = max(value_counts, key=value_counts.get)
            # print('transcript_id: ',transcript_id)
            # print('aoi_id_frequent: ',aoi_id_frequent)
            if aoi_id_frequent == None: continue
            aoi_material_match = self.aoi_material_dataset_agent[self.aoi_material_dataset_agent['slide_id_from_zero']==transcript_id]
            # print('aoi_material_match: ',aoi_material_match)
            aoi_material_match_item = aoi_material_match[aoi_material_match['aoi_id']==aoi_id_frequent]
            aoi_content = aoi_material_match_item['aoi_content'].values[0]
            # تعريب الجملة الوصفية
            transcripts_str += f'أثناء الشريحة: {transcript_id}, حرك الطالب الماوس أكثر على المحتوى أدناه: # {aoi_content} #. \n'
            valid_num += 1

        if valid_num == 0: return ''

        return transcripts_str

    def summarize_motor(self,motor_tuple_list):
        if len(motor_tuple_list) == 0 or motor_tuple_list == None: return ''
        # تعريب المقدمة
        transcripts_str = '# منطقة حركة الماوس (Mouse Move AOI) #: ملخص لمسار حركة الماوس الخاص بك أدناه. \n'
        valid_num = 0
        for d in motor_tuple_list:
            if d[0] in [-1,None] or len(d[0]) == 0: continue
            aoi_id_dict,transcript_id = d[0],d[1]
            # aoi_material_match = self.aoi_material_dataset_agent[self.aoi_material_dataset_agent['slide_id_from_zero']==transcript_id]
            aoi_id_dict_sorted = dict(sorted(aoi_id_dict.items()))
            action_trajectory = list(aoi_id_dict_sorted.values())

            # for aoi_id_each in action_trajectory:
            action_trajectory = [str(a_t) if a_t is not None else ' ' for a_t in action_trajectory]
            action_trajectory = ','.join(action_trajectory)
            # aoi_material_match_item = aoi_material_match[aoi_material_match['aoi_id']==aoaoi_id_eachi_id]
            # aoi_content = aoi_material_match_item['aoi_content'].values[0]
            # gaze_aoi_trajectory =
            # تعريب وصف المسار
            transcripts_str += f'أثناء الشريحة: {transcript_id}, مسار معرفات مناطق حركة الماوس (mouse moving AOI ID trajectory) هو: {action_trajectory}\n'
            valid_num += 1

        if valid_num == 0: return ''
        return transcripts_str

    def summarize_actions(self,memory_stream_list):
        if len(memory_stream_list) == 0 or memory_stream_list == None: return ''

        transcript_id_start = memory_stream_list[0]['transcript_id']
        transcript_id_end = memory_stream_list[-1]['transcript_id']
        # تعريب المقدمة
        summarized_actions_str = f'إليك مسار تجربتك من الشريحة {transcript_id_start} إلى الشريحة {transcript_id_end}. \n'
        summarized_actions_dict = {}
        for memory_element in memory_stream_list:
            transcript_id = memory_element['transcript_id']
            action_dict = memory_element['action']
            if len(action_dict) == 0: continue
            action_list = list(action_dict.keys())

            for action_name in action_list:
                action_value = memory_element['action'][action_name]
                if action_name in ['gaze_aoi_id','motor_aoi_id']:
                    if action_name not in list(summarized_actions_dict.keys()):
                        summarized_actions_dict[action_name] = [(action_value,transcript_id)]
                    else:
                        summarized_actions_dict[action_name].append((action_value,transcript_id))

                elif action_name in ['workload','curiosity','valid_focus','course_follow','engagement','confusion']:
                    if action_value == None or len(action_value) == 0:
                        action_trajectory = ''
                    else:
                        action_value_sorted = dict(sorted(action_value.items()))
                        action_trajectory = list(action_value_sorted.values())
                        action_trajectory = [str(round(a_t,2)) if a_t is not None else ' ' for a_t in action_trajectory]
                        action_trajectory = ','.join(action_trajectory)
                    if action_name not in list(summarized_actions_dict.keys()):
                        # تعريب الجملة الوصفية (مع إبقاء اسم الحالة الإنجليزية كما هو للدقة)
                        summarized_actions_dict[action_name] = f'مسار # {action_name} # الخاص بك هو: {action_trajectory},'
                    else:
                        summarized_actions_dict[action_name] = summarized_actions_dict[action_name] + action_trajectory + ', '

        action_list_sum = list(summarized_actions_dict.keys())

        for action in action_list_sum:
            if action not in ['gaze_aoi_id','motor_aoi_id']:
                summarized_actions_str += summarized_actions_dict[action] + '.\n'

        if 'gaze_aoi_id' in action_list_sum and len(summarized_actions_dict['gaze_aoi_id']) != 0 and summarized_actions_dict['gaze_aoi_id'] != None:
            sum_gaze_content = self.summarize_gaze(summarized_actions_dict['gaze_aoi_id'])
            summarized_actions_str += sum_gaze_content

        if 'motor_aoi_id' in action_list_sum and len(summarized_actions_dict['motor_aoi_id']) != 0 and summarized_actions_dict['motor_aoi_id'] != None:
            sum_motor_content = self.summarize_motor(summarized_actions_dict['motor_aoi_id'])
            summarized_actions_str += sum_motor_content

        return summarized_actions_str



    def summarize_aois(self,memory_stream_list):
        if len(memory_stream_list) == 0 or memory_stream_list == None: return ''
        transcripts_str = ''
        for memory_element in memory_stream_list:
            transcript_id = memory_element['transcript_id']
            # تعريب
            transcripts_str += f'#الشريحة: {transcript_id}#: \n'
            aoi_material_table = self.aoi_material_dataset_agent[self.aoi_material_dataset_agent['slide_id_from_zero']==transcript_id]
            transcripts_str += self._get_aoi_choice_str(aoi_material_table)
            transcripts_str += '\n'
        return transcripts_str

    def summarize_memory(self,memory_stream_list):
        transcript_id_start = memory_stream_list[0]['transcript_id']
        transcript_id_end = memory_stream_list[-1]['transcript_id']
        summarized_transcripts = '# ' + self.summarize_transcripts(memory_stream_list) + ' #.'
        summarized_actions = self.summarize_actions(memory_stream_list)
        summarized_aois = self.summarize_aois(memory_stream_list)

        # تعريب وصف الذاكرة المجمعة
        memory_string = (f'\n # إليك السجلات القديمة من الشريحة {transcript_id_start} إلى الشريحة {transcript_id_end} وتجربتك المقابلة لها. #\n' +
                        'محتويات الشرائح السابقة مدرجة أدناه: \n' + summarized_transcripts + '.\n\n' +
                        'محتويات مناطق الاهتمام (AOIs) في الشرائح السابقة مدرجة أدناه: \n' + summarized_aois + '.\n\n' +
                        'خلال شرائح الدورة هذه، أفعالك وحالاتك مدرجة أدناه: \n' + summarized_actions + '.\n\n')

        return memory_string

    def _generate_memory_string(self,memory_stream):
        # Warning: revising this function will affect both memory stream to agents as well as reflection & reasoning process
        if memory_stream == None or len(memory_stream) == 0:
            # تعريب
            return '\n\n لا يوجد سجل ذاكرة أو سجلات تاريخية لك.\n', ''

        # تعريب
        memory_string = '\n\n لقد مررت بهذه التجارب السابقة في الدورة الموضحة أدناه. \n'

        summarized_memories = self.summarize_memory(memory_stream)
        memory_string += summarized_memories

        if 'reflection' in list(memory_stream[-1].keys()):
            reflection_words = memory_stream[-1]['reflection']
            if len(reflection_words) != 0:
                # تعريب
                reflect_string = f'\n # التأمل الذاتي والتفكير المنطقي # الخاص بك لتاريخ الدورة وتجربتك السابقة هو أدناه: {reflection_words}. \n'
            else:
                reflect_string = ''
        else:
            reflect_string = ''

        return memory_string, reflect_string

    def _generate_memory_string_old(self,memory_stream,memory_hold_threshold = 10):
        # Warning: revising this function will affect both memory stream to agents as well as reflection & reasoning process
        if memory_stream == None or len(memory_stream) == 0:
            return 'لا يوجد سجل ذاكرة أو سجلات تاريخية لك.\n', ''

        memory_string = 'لقد مررت بهذه التجارب السابقة في الدورة الموضحة أدناه. \n'

        if len(memory_stream) > memory_hold_threshold:
            memory_stream_exact = memory_stream[-memory_hold_threshold:]
            summarized_memories = self.summarize_memory(memory_stream[:-memory_hold_threshold])
            memory_string += summarized_memories
        else:
            memory_stream_exact = memory_stream

        memory_string += f'\n # إليك أحدث سجلات النصوص وتجربتك. #\n'
        for memory_element in memory_stream_exact:
            transcript = memory_element['observation']
            if len(transcript)==0: continue
            transcript_id = memory_element['transcript_id']
            action_dict = memory_element['action']
            memory_string += f'\n # نص الدورة {transcript_id}#: # {transcript} #\n'
            if len(action_dict) == 0: continue
            memory_string += f'\n أفعالك لـ # النص {transcript_id}# هي أدناه:\n'
            action_list = list(action_dict.keys())

            aoi_material_match = self.aoi_material_dataset_agent[self.aoi_material_dataset_agent['slide_id_from_zero']==transcript_id]
            for action_name in action_list:
                action_value = memory_element['action'][action_name]
                if action_value == None:
                    continue

                # تعريب وصف الأفعال مع الحفاظ على مفاتيح مثل # Gaze Watch AOI #
                if action_name == 'gaze_aoi_id':
                    if action_value == -1:
                        memory_string += f'# Gaze Watch AOI #: لم تكن تنظر إلى أي مناطق اهتمام صالحة (valid AOIs) في الشرائح. \n'
                    else:
                        aoi_material_match_item = aoi_material_match[aoi_material_match['aoi_id']==action_value]
                        aoi_content = aoi_material_match_item['aoi_content'].values[0]
                        memory_string += f'# Gaze Watch AOI #: كنت تنظر إلى منطقة اهتمام واحدة (AOI) محتواها هو: # {aoi_content} #, \n'
                elif action_name == 'motor_aoi_id':
                    if action_value == -1:
                        memory_string += f'# Mouse Move AOI #: لم تكن تحرك الماوس على أي مناطق اهتمام صالحة في الشرائح. \n'
                    else:
                        aoi_material_match_item = aoi_material_match[aoi_material_match['aoi_id']==action_value]
                        aoi_content = aoi_material_match_item['aoi_content'].values[0]
                        memory_string += f'# Mouse Move AOI #: كنت تحرك الماوس لاستكشاف المحتويات في منطقة اهتمام واحدة (AOI) محتواها هو: # {aoi_content} #, \n'
                elif action_name == 'workload':
                    action_value = round(action_value,2)
                    memory_string += f'# WORKLOAD #: كان عبء العمل لديك # {action_value} #, \n'
                elif action_name == 'curiosity':
                    action_value = round(action_value,2)
                    memory_string += f'# CURIOSITY #: كان فضولك لاستكشاف محتويات الدورة # {action_value} #, \n'
                elif action_name == 'valid_focus':
                    action_value = round(action_value,2)
                    memory_string += f'# VALID FOCUS #: كان مدى تركيزك الفعلي # {action_value} #, \n'
                elif action_name == 'course_follow':
                    action_value = round(action_value,2)
                    memory_string += f'# COURSE FOLLOW #: كان مدى متابعتك للدورة # {action_value} #, \n'
                elif action_name == 'engagement':
                    action_value = round(action_value,2)
                    memory_string += f'# ENGAGEMENT #: كانت قيمة تفاعلك # {action_value} #, \n'
                elif action_name == 'confusion':
                    action_value = round(action_value,2)
                    memory_string += f'# CONFUSION #: كانت قيمة ارتباكك # {action_value} #, \n'

        # تعريب شرح المقاييس الإدراكية (مهم جداً للنموذج)
        cog_metric_str = ('\n\n لاحظ أن عبء العمل (WORKLOAD) هو قيمة رقمية من 0 (منخفض جداً) إلى 1 (مرتفع جداً) للإشارة إلى عبء العمل الذهني في الدورة. ' +
            'الفضول (CURIOSITY) هو قيمة رقمية من 0 (منخفض جداً) إلى 1 (مرتفع جداً) للإشارة إلى مدى فضولك لاستكشاف محتويات الدورة. ' +
            'التركيز الفعلي (VALID FOCUS) هو قيمة رقمية من 0 إلى 1 للإشارة إلى مدى قدرتك على التركيز على المحتويات الصالحة في الدورة. ' +
            'متابعة الدورة (COURSE FOLLOW) هي قيمة رقمية من 0 إلى 1 للإشارة إلى مدى قدرتك على متابعة وتيرة الدورة. ' +
            'التفاعل (ENGAGEMENT) هو قيمة صحيحة إما 0 (غير متفاعل) أو 1 (متفاعل). ' +
            'الارتباك (CONFUSION) هو قيمة صحيحة إما 0 (غير مرتبك) أو 1 (مرتبك) للإشارة إلى ما إذا كنت تشعر بالارتباك حول محتويات الدورة أم لا. \n\n' )

        memory_string = memory_string + cog_metric_str

        if 'reflection' in list(memory_stream[-1].keys()):
            reflection_words = memory_stream[-1]['reflection']
            if len(reflection_words) != 0:
                reflect_string = f'\n # التأمل الذاتي والتفكير المنطقي # الخاص بك لتاريخ الدورة وتجربتك السابقة هو أدناه: {reflection_words}. \n'
            else:
                reflect_string = ''
        else:
            reflect_string = ''

        return memory_string, reflect_string



    def load_memory_stream(self,current_transcript_id):

        if self.agent_config['memory_source'] == 'real':
            # إضافة ترميز utf-8 لقراءة العربية
            with open(self.user_memory_file, 'r', encoding='utf-8') as json_file:
                self.agent_real_memory_stream = json.load(json_file)
            retrieved_memory_stream = self.retrieve_memory(self.agent_real_memory_stream,current_transcript_id)

            return retrieved_memory_stream
        elif self.agent_config['memory_source'] == 'sim':
            # إضافة ترميز utf-8 لقراءة العربية
            with open(self.agent_memory_file, 'r', encoding='utf-8') as json_file:
                self.agent_sim_memory_stream = json.load(json_file)
            retrieved_memory_stream = self.retrieve_memory(self.agent_sim_memory_stream,current_transcript_id)

            return retrieved_memory_stream

    def add_to_agent_memory(self,memory_element):
        self._store_log('\n\n Adding new agent memory \n\n')
        # إضافة ترميز utf-8 للقراءة
        with open(self.agent_memory_file, 'r', encoding='utf-8') as json_file:
            self.agent_sim_memory_stream = json.load(json_file)

        self.agent_sim_memory_stream.append(memory_element)

        # إضافة ترميز utf-8 و ensure_ascii=False للكتابة (مهم جداً للعربية)
        with open(self.agent_memory_file, 'w', encoding='utf-8') as json_file:
            json.dump(self.agent_sim_memory_stream, json_file, indent=4, ensure_ascii=False)

    def add_to_user_memory(self,memory_element):
        self._store_log('\n\n Adding new user memory \n\n')
        # إضافة ترميز utf-8 للقراءة
        with open(self.user_memory_file, 'r', encoding='utf-8') as json_file:
            self.agent_real_memory_stream = json.load(json_file)

        self.agent_real_memory_stream.append(memory_element)

        # إضافة ترميز utf-8 و ensure_ascii=False للكتابة
        with open(self.user_memory_file, 'w', encoding='utf-8') as json_file:
            json.dump(self.agent_real_memory_stream, json_file, indent=4, ensure_ascii=False)

    # ------------------------------------------------------------------
    # دالة استدعاء النموذج المختار (تدعم 5 نماذج)
    # ------------------------------------------------------------------
    def _response_llm_gpt(self, message_list):
        """استدعاء النموذج المختار من الـ5 نماذج المتاحة"""
        return call_model(message_list, SELECTED_MODEL)

    # دوال وهمية (Dummy functions) لكي لا يتوقف البرنامج إذا حاول استدعاءها
    # لأنك حددت أنك تريد استخدام GPT فقط
    def _response_llm_gemini(self, messages):
        return ""

    def _response_llm_llama(self, sys_prompt, content_prompt, model_type):
        return ""

    def _get_real_gaze(self,during_table):
        if len(during_table) == 0: return None, (None, None)
        user_aoi_id = during_table['gaze_aoi_id'].mode().values[0]
        aoi_table = during_table[during_table['gaze_aoi_id']==user_aoi_id]
        if len(aoi_table) == 0: return None, (None, None)
        user_aoi_center_tuple = aoi_table['gaze_aoi_center_x_ratio'].values[0],aoi_table['gaze_aoi_center_y_ratio'].values[0]
        if user_aoi_id == -1: return None, (None, None)
        return user_aoi_id, user_aoi_center_tuple

    def _get_real_motor(self,during_table):
        if len(during_table) == 0: return None, (None, None)
        user_aoi_id = during_table['mouse_aoi_id'].mode().values[0]
        aoi_table = during_table[during_table['mouse_aoi_id']==user_aoi_id]
        if len(aoi_table) == 0: return None, (None, None)
        user_aoi_center_tuple = aoi_table['mouse_aoi_center_x_ratio'].values[0],aoi_table['mouse_aoi_center_y_ratio'].values[0]
        if user_aoi_id == -1: return None, (None, None)
        return user_aoi_id, user_aoi_center_tuple

    def _get_real_cognitive_state(self,during_table):
        if len(during_table) == 0: return None, None, None, None, None, None
        stationary_entropy_valid_table = during_table[during_table['gaze_entropy_stationary_norm']!=-1]
        transition_entropy_valid_table = during_table[during_table['gaze_entropy_transition_norm']!=-1]
        user_workload = None if len(stationary_entropy_valid_table) == 0 else round(stationary_entropy_valid_table['gaze_entropy_stationary_norm'].mean(),2)
        user_curiosity = None if len(transition_entropy_valid_table) == 0 else round(transition_entropy_valid_table['gaze_entropy_transition_norm'].mean(),2)

        user_valid_focus = round(during_table['valid_focus'].mean(),2) # average focus extent (0,1)
        user_course_follow = round(during_table['course_follow'].mean(),2) # average course follow extent (0,1)
        user_engagement = round(during_table['engagement'].mean(),2) # average course follow extent (0,1)
        user_confusion = round(during_table['confusion'].mean(),2) # average course follow extent (0,1)

        return user_workload, user_curiosity, user_valid_focus, user_course_follow, user_engagement, user_confusion

    def _calculate_distance(self, tuple1, tuple2):
        if tuple1 == (None,None) or tuple2 == (None,None) or None in tuple1 or None in tuple2:
            return None
        return math.sqrt((tuple1[0]-tuple2[0])**2 + (tuple1[1]-tuple2[1])**2)

    def _calculate_difference(self, val1, val2):
        if val1 is None or val2 is None:
            return None
        return abs(val1 - val2)

    def _simulate_gaze_motor_cog_question(self,example_demo_str,sim_strategy,retrieved_memory,transcript_id,transcript_material,aoi_material_item,user_table,question_item,user_answer_item):
        question_id_list = list(set(question_item['question_id']))
        question_id_list.sort()
        question_content_dict, choice_content_dict, correct_answer, user_answer = {}, {}, {}, {}
        for question_id in question_id_list:
            question_item_per = question_item[question_item['question_id']==question_id]
            question_content_dict[question_id] = question_item_per['question_content'].values[0]
            choice_content_dict[question_id] = question_item_per['choice_content'].values[0]
            correct_answer[question_id] = self.choice_map[question_item_per['correct_answer'].values[0]]

            user_answer_item_per = user_answer_item[user_answer_item['question_id']=='اختبار_س'+str(question_id)]

            if len(user_answer_item_per) == 0 or user_answer_item_per['choice'].values[0] is None or user_answer_item_per is None or user_answer_item_per['question_id'] is None:
                user_answer[question_id] = None
            else:
                try:
                    user_answer[question_id] = self.choice_map[user_answer_item_per['choice'].values[0]]
                except:
                    user_answer[question_id] = None

        sentence_id_list = list(set(aoi_material_item['transcript_id']))
        sentence_id_list.sort()

        agent_gaze_aoi_id, agent_gaze_aoi_center_tuple, agent_motor_aoi_id, agent_motor_aoi_center_tuple, agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion,agent_answer = self.action_gaze_mouse_cog_question_concise(sentence_id_list,example_demo_str,sim_strategy,retrieved_memory,transcript_id,transcript_material,aoi_material_item,question_content_dict,choice_content_dict)

        gaze_aoi_accuracy, gaze_aoi_distance, motor_aoi_accuracy, motor_aoi_distance, workload_diff, curiosity_diff, valid_focus_diff, follow_ratio_diff, engagement_accuracy, confusion_accuracy = {}, {}, {}, {}, {}, {}, {}, {}, {}, {}
        user_gaze_aoi_id,user_gaze_aoi_center_tuple,user_motor_aoi_id,user_motor_aoi_center_tuple = {}, {}, {}, {}
        user_workload,user_curiosity,user_valid_focus,user_course_follow,user_engagement,user_confusion = {}, {}, {}, {}, {}, {}
        choice_similarity_item,accuracy_similarity_item = {},{}
        for sentence_id in sentence_id_list:
            user_table_per = user_table[user_table['transcript_id']==sentence_id]
            user_gaze_table = user_table_per[user_table_per['gaze_aoi_id']!=-1]
            user_motor_table = user_table_per[user_table_per['mouse_aoi_id']!=-1]

            if agent_gaze_aoi_id == None or len(agent_gaze_aoi_id) == 0 or len(user_gaze_table) == 0:
                gaze_aoi_accuracy[sentence_id] = None
                gaze_aoi_distance[sentence_id] = None
                user_gaze_aoi_id[sentence_id] = None
                user_gaze_aoi_center_tuple[sentence_id] = (None,None)
                if self.agent_config['memory_source'] == 'real':
                    agent_gaze_aoi_id[sentence_id] = None
                    agent_gaze_aoi_center_tuple[sentence_id] = (None,None)

                if len(agent_gaze_aoi_id) == 0 or sentence_id not in list(agent_gaze_aoi_id.keys()):
                    agent_gaze_aoi_id[sentence_id] = None
                    agent_gaze_aoi_center_tuple[sentence_id] = (None,None)

            else:
                user_gaze_aoi_id_piece, user_gaze_aoi_center_tuple_piece = self._get_real_gaze(user_gaze_table)
                user_gaze_aoi_id[sentence_id] = user_gaze_aoi_id_piece
                user_gaze_aoi_center_tuple[sentence_id] = user_gaze_aoi_center_tuple_piece

                if sentence_id in list(agent_gaze_aoi_id.keys()):
                    gaze_aoi_accuracy[sentence_id] = 1 if agent_gaze_aoi_id[sentence_id] == user_gaze_aoi_id[sentence_id] else 0
                else:
                    agent_gaze_aoi_id[sentence_id] = None
                    gaze_aoi_accuracy[sentence_id] = None
                if sentence_id in list(agent_gaze_aoi_center_tuple.keys()):
                    gaze_aoi_distance[sentence_id] = self._calculate_distance(agent_gaze_aoi_center_tuple[sentence_id],user_gaze_aoi_center_tuple[sentence_id])
                else:
                    agent_gaze_aoi_center_tuple[sentence_id] = (None,None)
                    gaze_aoi_distance[sentence_id] = None

            if agent_motor_aoi_id == None or len(agent_motor_aoi_id) == 0 or len(user_motor_table) == 0:
                motor_aoi_accuracy[sentence_id] = None
                motor_aoi_distance[sentence_id] = None
                user_motor_aoi_id[sentence_id] = None
                user_motor_aoi_center_tuple[sentence_id] = (None,None)
                if self.agent_config['memory_source'] == 'real':
                    agent_motor_aoi_id[sentence_id] = None
                    agent_motor_aoi_center_tuple[sentence_id] = (None,None)
                if len(agent_motor_aoi_id) == 0 or sentence_id not in list(agent_motor_aoi_id.keys()):
                    agent_motor_aoi_id[sentence_id] = None
                    agent_motor_aoi_center_tuple[sentence_id] = (None,None)

            else:
                user_motor_aoi_id_piece, user_motor_aoi_center_tuple_piece = self._get_real_motor(user_motor_table)
                user_motor_aoi_id[sentence_id] = user_motor_aoi_id_piece
                user_motor_aoi_center_tuple[sentence_id] = user_motor_aoi_center_tuple_piece

                if sentence_id in list(agent_motor_aoi_id.keys()):
                    motor_aoi_accuracy[sentence_id] = 1 if agent_motor_aoi_id[sentence_id] == user_motor_aoi_id[sentence_id] else 0
                else:
                    agent_motor_aoi_id[sentence_id] = None
                    motor_aoi_accuracy[sentence_id] = None
                if sentence_id in list(agent_motor_aoi_center_tuple.keys()):
                    motor_aoi_distance[sentence_id] = self._calculate_distance(agent_motor_aoi_center_tuple[sentence_id],user_motor_aoi_center_tuple[sentence_id])
                else:
                    agent_motor_aoi_center_tuple[sentence_id] = (None,None)
                    motor_aoi_distance[sentence_id] = None

            user_workload_piece,user_curiosity_piece,user_valid_focus_piece,user_course_follow_piece,user_engagement_piece,user_confusion_piece = self._get_real_cognitive_state(user_table_per)
            user_workload[sentence_id] = user_workload_piece
            user_curiosity[sentence_id] = user_curiosity_piece
            user_valid_focus[sentence_id] = user_valid_focus_piece
            user_course_follow[sentence_id] = user_course_follow_piece
            user_engagement[sentence_id] = user_engagement_piece
            user_confusion[sentence_id] = user_confusion_piece

            if sentence_id in list(agent_workload.keys()):
                workload_diff[sentence_id] = self._calculate_difference(user_workload[sentence_id],agent_workload[sentence_id])
            else:
                agent_workload[sentence_id] = None
                workload_diff[sentence_id] = None
            if sentence_id in list(agent_curiosity.keys()):
                curiosity_diff[sentence_id] = self._calculate_difference(user_curiosity[sentence_id],agent_curiosity[sentence_id])
            else:
                agent_curiosity[sentence_id] = None
                curiosity_diff[sentence_id] = None
            if sentence_id in list(agent_valid_focus.keys()):
                valid_focus_diff[sentence_id] = self._calculate_difference(user_valid_focus[sentence_id],agent_valid_focus[sentence_id])
            else:
                agent_valid_focus[sentence_id] = None
                valid_focus_diff[sentence_id] = None
            if sentence_id in list(agent_course_follow.keys()):
                follow_ratio_diff[sentence_id] = self._calculate_difference(user_course_follow[sentence_id],agent_course_follow[sentence_id])
            else:
                agent_course_follow[sentence_id] = None
                follow_ratio_diff[sentence_id] = None
            if sentence_id in list(agent_engagement.keys()):
                engagement_accuracy[sentence_id] = self._calculate_difference(user_engagement[sentence_id],agent_engagement[sentence_id])
            else:
                agent_engagement[sentence_id] = None
                engagement_accuracy[sentence_id] = None
            if sentence_id in list(agent_confusion.keys()):
                confusion_accuracy[sentence_id] = self._calculate_difference(user_confusion[sentence_id],agent_confusion[sentence_id])
            else:
                agent_confusion[sentence_id] = None
                confusion_accuracy[sentence_id] = None

        for question_id in question_id_list:
            if question_id in list(agent_answer.keys()):
                user_accuracy_item = 1 if user_answer[question_id] == correct_answer[question_id] else 0
                agent_accuracy_item = 1 if agent_answer[question_id] == correct_answer[question_id] else 0
                choice_similarity_item[question_id] = 1 if user_answer[question_id] == agent_answer[question_id] else 0
                accuracy_similarity_item[question_id] = 1 if user_accuracy_item == agent_accuracy_item else 0
            else:
                agent_answer[question_id] = None
                choice_similarity_item[question_id] = None
                accuracy_similarity_item[question_id] = None

        return gaze_aoi_accuracy,gaze_aoi_distance,user_gaze_aoi_id,user_gaze_aoi_center_tuple,agent_gaze_aoi_id,agent_gaze_aoi_center_tuple,motor_aoi_accuracy,motor_aoi_distance,user_motor_aoi_id,user_motor_aoi_center_tuple,agent_motor_aoi_id,agent_motor_aoi_center_tuple,workload_diff,curiosity_diff,valid_focus_diff,follow_ratio_diff,engagement_accuracy,confusion_accuracy,user_workload,user_curiosity,user_valid_focus,user_course_follow,user_engagement,user_confusion,agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion,choice_similarity_item,accuracy_similarity_item,user_answer,agent_answer,correct_answer

    def _get_transcript_str(self,tiny_transcript_id_list):
        transcript_material = ''
        for tiny_transcript_id in tiny_transcript_id_list:
            tiny_content = self.transcript_dict_agent[int(tiny_transcript_id)]['content']
            # تم تعريب النص هنا
            transcript_material += (f'\n # معرف النص: {tiny_transcript_id} #: المحتوى: # {tiny_content} #.')
        return transcript_material

    def agent_run_during_class(self):
        print('self.exist_simulation_transcript_id_list: ',self.exist_simulation_transcript_id_list)
        for transcript_id in self.transcript_id_list_simulation:
            if len(self.exist_simulation_transcript_id_list) != 0:
                if transcript_id in self.exist_simulation_transcript_id_list: continue
                self.exist_simulation_transcript_id_list.sort()
                if transcript_id != (self.exist_simulation_transcript_id_list[-1]+1): continue

            print(f'similating user {self.agent_id} in slide: {transcript_id}')
            self._store_log('\n بدء المحاكاة للشريحة '+str(transcript_id)+'-'*20+'\n\n')
            # transcript_material = self.transcript_dict_agent[transcript_id]['content']
            course_material_item = self.course_dataset_agent[self.course_dataset_agent['slide_id_from_zero']==transcript_id]
            tiny_transcript_id_list = list(set(course_material_item['transcript_id']))
            tiny_transcript_id_list.sort()
            transcript_material = self._get_transcript_str(tiny_transcript_id_list)

            aoi_material_item = self.aoi_material_dataset_agent[self.aoi_material_dataset_agent['slide_id_from_zero']==transcript_id]
            during_item = self.during_dataset_agent[self.during_dataset_agent['slide_id_from_zero']==transcript_id]

            # question_id = question_id_map_slide_dict['video_'+str(self.video_id)][transcript_id][0]
            # question_item = self.question_dataset_agent[self.question_dataset_agent['question_id']==question_id]
            # user_answer_item = self.student_answer_item_dataset_agent[self.student_answer_item_dataset_agent['question_id']=='test_q'+str(question_id)]

            question_id_list = question_id_map_slide_dict['video_'+str(self.video_id)][transcript_id]
            question_id_ext_list = ['اختبار_س' + str(q_id) for q_id in question_id_list]  # تم التصحيح: استخدام 'اختبار_س' بدلاً من 'test_q'
            question_item = self.question_dataset_agent[self.question_dataset_agent['question_id'].isin(question_id_list)]
            user_answer_item = self.student_answer_item_dataset_agent[self.student_answer_item_dataset_agent['question_id'].isin(question_id_ext_list)]

            retrieved_memory_stream = self.load_memory_stream(transcript_id)
            retrieved_memory_stream_no_reflect_str,reflect_string = self._generate_memory_string(retrieved_memory_stream)
            retrieved_memory_stream_str = retrieved_memory_stream_no_reflect_str + reflect_string

            if self.agent_config['example_demo'] == 'yes':
                example_demo_str = self.obtain_example_demo_str(transcript_id,question_id_list)
            else:
                example_demo_str = ''

            # gaze_aoi_accuracy,gaze_aoi_distance,user_gaze_aoi_id,user_gaze_aoi_center_tuple,agent_gaze_aoi_id,agent_gaze_aoi_center_tuple = self._simulate_gaze(retrieved_memory_stream_str,transcript_id,transcript_material,aoi_material_item,during_item)
            # motor_aoi_accuracy,motor_aoi_distance,user_motor_aoi_id,user_motor_aoi_center_tuple,agent_motor_aoi_id,agent_motor_aoi_center_tuple = self._simulate_motor(retrieved_memory_stream_str,transcript_id,transcript_material,aoi_material_item,during_item)
            # workload_diff,curiosity_diff,valid_focus_diff,follow_ratio_diff,engagement_accuracy,confusion_accuracy,user_workload,user_curiosity,user_valid_focus,user_course_follow,user_engagement,user_confusion,agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion = self._simulate_cognitive_state(retrieved_memory_stream_str,transcript_id,transcript_material,aoi_material_item,during_item)
            sim_strategy = self.agent_config['sim_strategy']

            gaze_aoi_accuracy,gaze_aoi_distance,user_gaze_aoi_id,user_gaze_aoi_center_tuple,agent_gaze_aoi_id,agent_gaze_aoi_center_tuple,motor_aoi_accuracy,motor_aoi_distance,user_motor_aoi_id,user_motor_aoi_center_tuple,agent_motor_aoi_id,agent_motor_aoi_center_tuple,workload_diff,curiosity_diff,valid_focus_diff,follow_ratio_diff,engagement_accuracy,confusion_accuracy,user_workload,user_curiosity,user_valid_focus,user_course_follow,user_engagement,user_confusion,agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion,choice_similarity_item,accuracy_similarity_item,user_answer,agent_answer,correct_answer = self._simulate_gaze_motor_cog_question(example_demo_str,sim_strategy,retrieved_memory_stream_str,transcript_id,transcript_material,aoi_material_item,during_item,question_item,user_answer_item)

            sentence_id_list = list(set(aoi_material_item['transcript_id']))
            sentence_id_list.sort()


            if self.agent_config['memory_source'] == 'real':
                user_action_dict = {'gaze_aoi_id': user_gaze_aoi_id, 'motor_aoi_id': user_motor_aoi_id, 'workload': user_workload,'curiosity': user_curiosity,'valid_focus': user_valid_focus,'course_follow': user_course_follow,'engagement': user_engagement,'confusion': user_confusion}
                memory_element = {'transcript_id':transcript_id,'observation':transcript_material,'action':user_action_dict}
                if self.agent_config['reflection_choice'] == 'yes':
                    memory_element['reflection'] = self.reflect_reason(retrieved_memory_stream_no_reflect_str)
                self.add_to_user_memory(memory_element)
            if self.agent_config['memory_source'] == 'sim':
                agent_action_dict = {'gaze_aoi_id': agent_gaze_aoi_id, 'motor_aoi_id': agent_motor_aoi_id, 'workload': agent_workload, 'curiosity': agent_curiosity, 'valid_focus': agent_valid_focus, 'course_follow': agent_course_follow, 'engagement': agent_engagement, 'confusion': agent_confusion}
                memory_element = {'transcript_id':transcript_id,'observation':transcript_material,'action':agent_action_dict}
                if self.agent_config['reflection_choice'] == 'yes':
                    memory_element['reflection'] = self.reflect_reason(retrieved_memory_stream_no_reflect_str)
                self.add_to_agent_memory(memory_element)

            for sentence_id in sentence_id_list:
                user_dur_result_list = [user_gaze_aoi_id[sentence_id],user_gaze_aoi_center_tuple[sentence_id][0],user_gaze_aoi_center_tuple[sentence_id][1],user_motor_aoi_id[sentence_id],user_motor_aoi_center_tuple[sentence_id][0],user_motor_aoi_center_tuple[sentence_id][1],user_workload[sentence_id],user_curiosity[sentence_id],user_valid_focus[sentence_id],user_course_follow[sentence_id],user_engagement[sentence_id],user_confusion[sentence_id]]
                try:
                    agent_dur_result_list = [agent_gaze_aoi_id[sentence_id],agent_gaze_aoi_center_tuple[sentence_id][0],agent_gaze_aoi_center_tuple[sentence_id][1],agent_motor_aoi_id[sentence_id],agent_motor_aoi_center_tuple[sentence_id][0],agent_motor_aoi_center_tuple[sentence_id][1],agent_workload[sentence_id],agent_curiosity[sentence_id],agent_valid_focus[sentence_id],agent_course_follow[sentence_id],agent_engagement[sentence_id],agent_confusion[sentence_id]]
                except:
                    print(sentence_id,agent_gaze_aoi_id,agent_gaze_aoi_center_tuple,agent_motor_aoi_id,agent_motor_aoi_center_tuple,agent_workload,agent_curiosity,agent_valid_focus,agent_course_follow,agent_engagement,agent_confusion)
                    assert 1==0
                metric_ind_dur_result_list = [gaze_aoi_accuracy[sentence_id],gaze_aoi_distance[sentence_id],motor_aoi_accuracy[sentence_id],motor_aoi_distance[sentence_id],workload_diff[sentence_id],curiosity_diff[sentence_id],valid_focus_diff[sentence_id],follow_ratio_diff[sentence_id],engagement_accuracy[sentence_id],confusion_accuracy[sentence_id]]

                # تعديل: إضافة الترميز الصحيح utf-8-sig
                with open(self.sim_result_dur_path,'a+', encoding='utf-8-sig') as fwrite_ind:
                    write_list = self.sim_config_set + [str(self.agent_id),str(transcript_id),str(sentence_id)] + user_dur_result_list + agent_dur_result_list + metric_ind_dur_result_list
                    fwrite_ind.write(self._list_to_str_line(write_list))

            for question_id in question_id_list:
                metric_ind_question_result_list = [choice_similarity_item[question_id],accuracy_similarity_item[question_id]]
                # تعديل: إضافة الترميز الصحيح utf-8-sig
                with open(self.sim_result_post_path,'a+', encoding='utf-8-sig') as fwrite_ind:
                    write_list = self.sim_config_set + [str(self.agent_id),str(question_id)] + [user_answer[question_id],agent_answer[question_id],correct_answer[question_id]] + metric_ind_question_result_list
                    fwrite_ind.write(self._list_to_str_line(write_list))

            self.exist_simulation_transcript_id_list.append(transcript_id)

    def agent_run_all(self):
        self._store_log('\n\n بدء المحاكاة للوكيل '+str(self.agent_id)+' في الفيديو: '+str(self.video_id)+'-'*40+'\n\n')
        self.instantiate_profile()
        self.instantiate_memory()

        self.agent_run_during_class()
        # self.agent_run_post_class()

    def _store_log(self,input_string,color=None, attrs=None, print=False):
        # تعديل: إضافة الترميز الصحيح utf-8
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(input_string + '\n')
            f.flush()
        if(print):
            cprint(input_string, color=color, attrs=attrs)

    def _calculate_mse(self,diff_list):
        # diff_list_filter = self._remove_none_from_list(diff_list)
        return sum(x**2 for x in diff_list) / len(diff_list)

    def _calculate_difference(self,truth,prediction):
        if truth == None or prediction == None:
            return None
        return abs(truth-prediction)

    def _calculate_same(self,user_truth,agent_prediction):
        if user_truth == None or agent_prediction == None:
            return None
        if user_truth == agent_prediction:
            return 1
        else:
            return 0

    # دالة حساب المسافة (موجودة هنا أيضاً، ممتاز)
    def _calculate_distance(self,point_1_tuple,point_2_tuple):
        if point_1_tuple == None or point_2_tuple == None or point_1_tuple[0] == None or point_1_tuple[1] == None or point_2_tuple[0] == None or point_2_tuple[1] == None:
            return None
        return math.sqrt((point_1_tuple[0]-point_2_tuple[0])*(point_1_tuple[0]-point_2_tuple[0])+(point_1_tuple[1]-point_2_tuple[1])*(point_1_tuple[1]-point_2_tuple[1]))

    def _calculate_accuracy(self,true_labels,predicted_labels):
        if len(true_labels) != len(predicted_labels):
            raise ValueError("Input lists must have the same length.")
        compare_list = []
        for comi in range(true_labels):
            if true_labels[comi] != None and predicted_labels[comi] != None:
                compare_list.append(int(true_labels[comi]==predicted_labels[comi]))
            else:
                compare_list.append(None)
        # compare_list = [int(true == pred) for true, pred in zip(true_labels, predicted_labels)]
        correct_predictions = sum(compare_list)
        accuracy = correct_predictions / len(compare_list)
        return accuracy,compare_list

    def _remove_none_from_list(self,input_list):
        filtered_list = [value for value in input_list if value is not None]
        return filtered_list

    def _list_to_str_line(self,input_list):
        return ','.join(str(u_value) for u_value in input_list)+'\n'





# -----------------------------------------------------------------------------
# هنا ينتهي كلاس Avatar (لاحظ أن الكود التالي يبدأ من بداية السطر بدون مسافات)
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# تشغيل التجربة (Run Experiment)
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# تشغيل التجربة (Run Experiment) - نسخة مرنة (10 طلاب أو الكل)
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# تشغيل التجربة الشاملة (Full Experiment on ALL Students)
# -----------------------------------------------------------------------------

import nest_asyncio
nest_asyncio.apply()

def simulate_student(student_id, agent_config):
    print(f'Starting simulation for student: {student_id}')
    try:
        avatar_item = Avatar(agent_config, agent_id=student_id)
        avatar_item.agent_run_all()
    except Exception as e:
        print(f"Skipping Student {student_id} due to error: {e}")

async def run_exp(agent_config, user_start_index=None, user_end_index=None):

    print("جاري قراءة الملفات لتحديد جميع الطلاب المتاحين...")

    # 1. قراءة الملفات لتحديد القائمة الكاملة
    # -----------------------------------------------------------
    try:
        # أ) ملف السلوك (الأهم): يستخدم الفاصلة العادية (,)
        behavior_table = pd.read_csv(agent_config['dataset_path']+'/during_behavior_slide.csv', encoding='utf-8-sig', sep=',')
        valid_ids_behavior = set(behavior_table['student_id'])

        # ب) ملف الديموغرافيا: يستخدم الفاصلة المنقوطة (;)
        demo_table = pd.read_csv(agent_config['dataset_path']+'/student_demo.csv', encoding='utf-8-sig', sep=';')
        valid_ids_demo = set(demo_table['student_id'])

        # ج) التقاطع: نختار الطلاب الموجودين في الملفين فقط (لضمان وجود بيانات لهم)
        all_valid_students = list(valid_ids_behavior.intersection(valid_ids_demo))
        all_valid_students.sort()

    except Exception as e:
        print(f"Error reading dataset files: {e}")
        return

    # 2. تحديد قائمة الطلاب (تصفية حسب النطاق المطلوب)
    # -----------------------------------------------------------

    # تصفية الطلاب بناءً على user_start_index و user_end_index
    if user_start_index is not None and user_end_index is not None:
        # استخدام النطاق كمؤشر (index) وليس كمعرف (ID)
        # مثال: user_start_index=0, user_end_index=29 يعني أول 30 طالب
        student_list_select = all_valid_students[user_start_index:user_end_index+1]
    else:
        # إذا لم يتم تحديد نطاق، استخدم جميع الطلاب
        student_list_select = all_valid_students

    # إذا أردت مستقبلاً تجربة طالب واحد فقط، ألغِ تعليق السطر التالي:
    # student_list_select = [136]

    # -----------------------------------------------------------

    print('=' * 60)
    print(f'🚀 وضع التشغيل الشامل (All Students Mode)')
    print(f'سيتم محاكاة عدد: {len(student_list_select)} طالب.')
    print(f'القائمة الكاملة: {student_list_select}')
    print('=' * 60)

    loop = asyncio.get_event_loop()

    # عدد العمليات المتوازية
    # نصيحة: إذا واجهت أخطاء (Rate Limit Error 429)، قلل الرقم إلى 3 أو 2
    executor = ThreadPoolExecutor(max_workers=5)

    tasks = []

    for student_id in student_list_select:
        # إنشاء المجلدات الضرورية
        if os.path.exists(agent_config['result_path']) == False:
            os.makedirs(agent_config['result_path'])

        root_folder = agent_config['result_path'] + '/' + agent_config['memory_source'] + '_' + agent_config['forget_effect'] + '_reflect-' + agent_config['reflection_choice'] + '_' + agent_config['memory_component_choice'] + '_' + agent_config['sim_strategy'] + '_example-' + agent_config['example_demo'] + '_' + str(agent_config['gpt_type'])

        if os.path.exists(root_folder) == False:
            os.makedirs(root_folder)
        if os.path.exists(root_folder + '/log') == False:
            os.makedirs(root_folder + '/log')
        if os.path.exists(root_folder + '/agent_memory') == False:
            os.makedirs(root_folder + '/agent_memory')
        if os.path.exists(root_folder + '/user_memory') == False:
            os.makedirs(root_folder + '/user_memory')

        task = loop.run_in_executor(executor, simulate_student, student_id, agent_config)
        tasks.append(task)

    await asyncio.gather(*tasks)


# ================================================================================
# ===== 🚀 دالة تشغيل جميع التجارب للـ5 نماذج =====
# ================================================================================

# قائمة النماذج التي سيتم تشغيلها (5 نماذج: GPT-3.5, GPT-4o Mini, Llama 3.1, Qwen2.5)
ALL_MODELS = [2, 3, 4, 5]  # GPT-3.5, GPT-4o Mini, Llama 3.1, Qwen2.5 (بدون النموذج السادس)

# قائمة التجارب (real = التنبؤ، sim = التوليد)
EXPERIMENTS = ['real', 'sim']  # التجربتين: التنبؤ والتوليد

async def run_all_experiments(user_start_index=1000, user_end_index=1001):
    """
    دالة لتشغيل جميع التجارب تلقائياً:
    - 5 نماذج × 2 تجربة = 10 تجارب
    - كل تجربة تحاكي الطلاب المحددين

    ملاحظة: التجربة الثانية (sim) ستستخدم أول 300 طالب فقط،
    بينما التجربة الأولى (real) ستستخدم جميع الطلاب المحددين.
    """
    global SELECTED_MODEL

    # الإعدادات الأساسية
    base_config = {
        'dataset_path': 'dataset/',
        'result_path': 'dataset/simulation/full_experiment_300_students',
        'sim_strategy': 'standard_cog',
        'example_demo': 'no',
        'reflection_choice': 'no',
        'forget_effect': 'only_recent_one',
        'memory_component_choice': 'KM+PM+MM+CM',
        'example_user_dict': {
            'video_1': [167, 179, 153],
            'video_2': [590, 321, 327],
            'video_3': [366, 349, 342],
            'video_4': [436, 729, 696],
            'video_5': [798, 789, 507]
        },
    }

    execution_log = []

    # 🗑️ حذف جميع النتائج القديمة تلقائياً قبل البدء
    print("="*80)
    print("🗑️  حذف النتائج القديمة...")
    result_path = base_config['result_path']
    if os.path.exists(result_path):
        import shutil
        for item in os.listdir(result_path):
            item_path = os.path.join(result_path, item)
            try:
                if os.path.isdir(item_path):
                    shutil.rmtree(item_path)
                    print(f"   ✓ تم حذف المجلد: {item}")
                elif os.path.isfile(item_path):
                    os.remove(item_path)
                    print(f"   ✓ تم حذف الملف: {item}")
            except Exception as e:
                print(f"   ⚠️ خطأ في حذف {item}: {e}")
        print("✅ تم حذف جميع النتائج القديمة بنجاح!")
    else:
        print("✅ لا توجد نتائج قديمة للحذف.")
    print("="*80)

    print("="*80)
    print("🚀 بدء التشغيل التلقائي للنماذج المتعددة")
    print("="*80)
    print(f"📊 عدد النماذج: {len(ALL_MODELS)}")
    print(f"📊 عدد التجارب: {len(EXPERIMENTS)}")
    print(f"👥 عدد الطلاب: {user_end_index - user_start_index + 1}")
    print(f"⚡ إجمالي التجارب: {len(ALL_MODELS) * len(EXPERIMENTS)} تجربة")
    print("="*80)

    start_time = time.time()

    for model_num in ALL_MODELS:
        SELECTED_MODEL = model_num  # تحديث المتغير العام
        model_name = MODEL_DISPLAY[model_num]

        for experiment_type in EXPERIMENTS:
            exp_name = "التجربة 1 (التنبؤ)" if experiment_type == 'real' else "التجربة 2 (التوليد)"

            print("\n" + "="*80)
            print(f"🤖 النموذج: {model_name}")
            print(f"📋 {exp_name}")
            print(f"⏰ الوقت: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)

            # إنشاء الإعدادات لهذه التجربة
            current_config = base_config.copy()
            current_config['memory_source'] = experiment_type
            current_config['gpt_type'] = model_num

            exp_start_time = time.time()

            try:
                # تحديد نطاق الطلاب حسب نوع التجربة
                # التجربة الأولى (real): جميع الطلاب المحددين
                # التجربة الثانية (sim): أول 300 طالب فقط
                if experiment_type == 'sim':
                    # للتجربة الثانية: نستخدم فقط أول 300 طالب (من 0 إلى 299)
                    exp_start = user_start_index
                    exp_end = min(299, user_end_index)  # أقصى حد هو 299 (أي 300 طالب)
                    print(f"   📊 عدد الطلاب للتوليد: {exp_end - exp_start + 1}")
                else:
                    # للتجربة الأولى: نستخدم جميع الطلاب
                    exp_start = user_start_index
                    exp_end = user_end_index
                    print(f"   📊 عدد الطلاب للمحاكاة: {exp_end - exp_start + 1}")

                # تشغيل التجربة مع نطاق الطلاب المحدد
                await run_exp(current_config, exp_start, exp_end)

                exp_duration = time.time() - exp_start_time
                status = "✅ نجح"

                print(f"\n✅ اكتملت التجربة في {exp_duration/60:.2f} دقيقة")

            except Exception as e:
                exp_duration = time.time() - exp_start_time
                status = f"❌ فشل: {str(e)}"
                print(f"\n❌ فشلت التجربة: {e}")

            # حفظ السجل
            execution_log.append({
                'model_num': model_num,
                'model_name': model_name,
                'experiment': experiment_type,
                'experiment_name': exp_name,
                'status': status,
                'duration_minutes': exp_duration / 60,
                'num_students': user_end_index - user_start_index + 1,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })

    total_duration = time.time() - start_time

    print("\n" + "="*80)
    print("🎉 اكتمل التشغيل التلقائي الشامل!")
    print("="*80)
    print(f"⏱️ إجمالي الوقت: {total_duration/3600:.2f} ساعة")
    print(f"📊 عدد التجارب: {len(execution_log)}")
    print("="*80)

    # حفظ السجل في ملف
    log_df = pd.DataFrame(execution_log)
    log_path = 'dataset/simulation/full_experiment_300_students/execution_log.csv'
    log_df.to_csv(log_path, index=False, encoding='utf-8-sig')
    print(f"💾 تم حفظ سجل التشغيل في: {log_path}")

    return execution_log

# ================================================================================

# إعدادات المحاكاة (للتشغيل اليدوي لنموذج واحد)
simulation_config_1 = {
    'dataset_path': 'dataset/',
    'result_path': 'dataset/simulation/full_experiment_300_students',
    'memory_source': 'sim',
    'sim_strategy': 'standard_cog',
    'example_demo': 'no',
    'gpt_type': 3,  # GPT-4o Mini افتراضياً
    'reflection_choice': 'no',
    'forget_effect': 'only_recent_one',
    'memory_component_choice': 'KM+PM+MM+CM',
    'example_user_dict': {'video_1': [167, 179, 153], 'video_2': [590, 321, 327], 'video_3': [366, 349, 342], 'video_4': [436, 729, 696], 'video_5': [798, 789, 507]},
}

# ================================================================================
# ===== 🎯 التشغيل الرئيسي =====
# ================================================================================

if __name__ == "__main__":
    # للتشغيل اليدوي لنموذج واحد:
    # asyncio.run(run_exp(simulation_config_1))

    # ═══════════════════════════════════════════════════════════════════════════
    # 🚀 التشغيل الكامل للتجربة: محاكاة جميع الطلاب
    # ═══════════════════════════════════════════════════════════════════════════
    # التجربة الأولى (real): محاكاة جميع الطلاب الحقيقيين (~310 طالب)
    # التجربة الثانية (sim): توليد 300 طالب افتراضي
    #
    # user_start_index=0, user_end_index=309 = أول 310 طالب (جميع الطلاب الحقيقيين)
    # user_start_index=0, user_end_index=299 = أول 300 طالب (للطلاب الافتراضيين)
    # ═══════════════════════════════════════════════════════════════════════════

    asyncio.run(run_all_experiments(user_start_index=0, user_end_index=309))


import pandas as pd
import os
import glob

# 1. تحديد مسار المجلد الذي يحتوي على النتائج
# بما أن اسم المجلد طويل ومتغير، سنبحث عنه تلقائياً
base_path = 'dataset/simulation/test'
search_path = os.path.join(base_path, '*', 'log', '*_result_ind_dur.csv')
files_dur = glob.glob(search_path)

if not files_dur:
    print("لم يتم العثور على ملفات نتائج! تأكد من أن المحاكاة انتهت بنجاح.")
else:
    # نأخذ أول ملف نجده (للطالب 1000)
    file_dur = files_dur[0]
    # ملف الأسئلة يكون في نفس المجلد ولكن ينتهي بـ _post.csv
    file_post = file_dur.replace('_result_ind_dur.csv', '_result_ind_post.csv')

    print(f"جاري تحليل النتائج من الملف:\n{file_dur}\n")
    print("-" * 50)

    # --- حساب دقة السلوك (نظر، ماوس، حالات إدراكية) ---
    try:
        df_dur = pd.read_csv(file_dur, encoding='utf-8-sig')

        # 1. دقة النظر (Gaze Accuracy)
        if 'gaze_aoi_accuracy' in df_dur.columns:
            gaze_acc = df_dur['gaze_aoi_accuracy'].mean() * 100
            print(f"👁️  دقة النظر (Gaze Accuracy): {gaze_acc:.2f}%")

        # 2. دقة حركة الماوس (Motor Accuracy)
        if 'motor_aoi_accuracy' in df_dur.columns:
            motor_acc = df_dur['motor_aoi_accuracy'].mean() * 100
            print(f"🖱️  دقة الماوس (Motor Accuracy): {motor_acc:.2f}%")

        # 3. دقة التفاعل (Engagement Accuracy)
        if 'engagement_accuracy' in df_dur.columns:
            eng_acc = (1 - df_dur['engagement_accuracy'].mean()) * 100 # لأن القيمة هي الفرق (Difference)
            print(f"🧠  دقة التفاعل (Engagement Match): {eng_acc:.2f}%")

    except Exception as e:
        print(f"خطأ في قراءة ملف السلوك: {e}")

    print("-" * 50)

    # --- حساب دقة الأسئلة (Questions) ---
    try:
        if os.path.exists(file_post):
            df_post = pd.read_csv(file_post, encoding='utf-8-sig')

            # 1. التشابه مع إجابة الطالب الحقيقي (Choice Similarity)
            # هل اختار الوكيل نفس إجابة الطالب الحقيقي؟
            if 'choice_similarity' in df_post.columns:
                sim_acc = df_post['choice_similarity'].mean() * 100
                print(f"✅  تطابق الإجابة مع الطالب الحقيقي: {sim_acc:.2f}%")

            # 2. هل كانت إجابة الوكيل صحيحة علمياً؟ (Agent Correctness)
            # نقارن إجابة الوكيل (agent_answer) مع الإجابة الصحيحة (correct_answer)
            # نحتاج تحويل الأعمدة لأرقام للمقارنة
            correct_count = 0
            total_count = len(df_post)
            for index, row in df_post.iterrows():
                if row['agent_answer'] == row['correct_answer']:
                    correct_count += 1

            if total_count > 0:
                scientific_acc = (correct_count / total_count) * 100
                print(f"📚  نسبة الإجابات الصحيحة علمياً: {scientific_acc:.2f}%")

        else:
            print("ملف نتائج الأسئلة غير موجود.")

    except Exception as e:
        print(f"خطأ في قراءة ملف الأسئلة: {e}")




import pandas as pd
import os
import glob
import numpy as np

# 1. تحديد مسار المجلد الجديد الذي حددناه في الخطوة السابقة
base_path = 'dataset/simulation/full_experiment_300_students'

# البحث عن كل ملفات النتائج السلوكية (dur) داخل المجلدات الفرعية
search_path = os.path.join(base_path, '*', 'log', '*_result_ind_dur.csv')
files_dur = glob.glob(search_path)

if not files_dur:
    print(f"❌ لم يتم العثور على ملفات نتائج في المسار: {base_path}")
    print("تأكد أن المحاكاة انتهت وأن اسم المجلد مطابق.")
else:
    print(f"✅ تم العثور على {len(files_dur)} ملفات نتائج للطلاب.")
    print("=" * 60)

    # قوائم لتخزين المتوسطات لحساب المعدل العام لاحقاً
    all_gaze_acc = []
    all_motor_acc = []
    all_sim_acc = []
    all_sci_acc = []

    for file_dur in files_dur:
        # استخراج رقم الطالب من اسم الملف
        file_name = os.path.basename(file_dur)
        student_id = file_name.split('_')[0]

        # تحديد ملف الأسئلة المقابل
        file_post = file_dur.replace('_result_ind_dur.csv', '_result_ind_post.csv')

        print(f"📊 تحليل نتائج الطالب: {student_id}")

        # --- 1. تحليل السلوك (Behavior) ---
        try:
            df_dur = pd.read_csv(file_dur, encoding='utf-8-sig')

            # دقة النظر
            gaze = df_dur['gaze_aoi_accuracy'].mean() * 100
            all_gaze_acc.append(gaze)

            # دقة الماوس
            motor = df_dur['motor_aoi_accuracy'].mean() * 100
            all_motor_acc.append(motor)

            # دقة التفاعل
            # (engagement_accuracy) في الكود هي الفرق (Diff)، لذا الدقة هي 1 - الفرق
            if 'engagement_accuracy' in df_dur.columns:
                eng = (1 - df_dur['engagement_accuracy'].mean()) * 100
            else:
                eng = 0

            print(f"   👁️  دقة النظر: {gaze:.2f}%")
            print(f"   🖱️  دقة الماوس: {motor:.2f}%")
            print(f"   🧠  دقة التفاعل: {eng:.2f}%")

        except Exception as e:
            print(f"   ⚠️ خطأ في قراءة ملف السلوك: {e}")

        # --- 2. تحليل الأسئلة (Questions) ---
        try:
            if os.path.exists(file_post):
                df_post = pd.read_csv(file_post, encoding='utf-8-sig')

                # تطابق الإجابة مع الطالب الحقيقي
                sim = df_post['choice_similarity'].mean() * 100
                all_sim_acc.append(sim)

                # الإجابة الصحيحة علمياً
                # نحتاج لمقارنة agent_answer مع correct_answer
                # ملاحظة: أحياناً تكون الإجابات مخزنة كأرقام 0,1,2 وأحياناً نصوص A,B,C حسب الكود
                correct_count = 0
                for idx, row in df_post.iterrows():
                    # تحويل القيم لنصوص للمقارنة الآمنة
                    if str(row['agent_answer']).strip() == str(row['correct_answer']).strip():
                        correct_count += 1

                sci = (correct_count / len(df_post)) * 100 if len(df_post) > 0 else 0
                all_sci_acc.append(sci)

                print(f"   ✅  تطابق الإجابة مع الطالب: {sim:.2f}%")
                print(f"   📚  الإجابات الصحيحة علمياً: {sci:.2f}%")
            else:
                print("   ⚠️ ملف الأسئلة غير موجود.")
        except Exception as e:
            print(f"   ⚠️ خطأ في قراءة ملف الأسئلة: {e}")

        print("-" * 30)

    # --- الخلاصة النهائية (Overall Average) ---
    print("\n" + "=" * 60)
    print("📈 المتوسط العام لجميع الطلاب (Overall Performance)")
    print("=" * 60)
    print(f"👁️  متوسط دقة النظر: {np.nanmean(all_gaze_acc):.2f}%")
    print(f"🖱️  متوسط دقة الماوس: {np.nanmean(all_motor_acc):.2f}%")
    print(f"✅  متوسط تطابق الإجابات مع البشر: {np.nanmean(all_sim_acc):.2f}%")
    print(f"📚  متوسط الإجابات الصحيحة علمياً: {np.nanmean(all_sci_acc):.2f}%")
    print("=" * 60)


# ═══════════════════════════════════════════════════════════════════════════
# 📊 تحليل شامل لجميع النتائج (Comprehensive Analysis for All Models)
# ═══════════════════════════════════════════════════════════════════════════

def analyze_all_experiments():
    """
    تحليل شامل لجميع النتائج من جميع النماذج والتجارب
    يعرض:
    - مقارنة بين النماذج المختلفة (5 نماذج)
    - مقارنة بين التجربتين (real vs sim)
    - أفضل نموذج لكل مقياس
    - حفظ النتائج في ملف CSV
    """
    print("\n\n" + "═"*100)
    print("📊 تحليل شامل لجميع نتائج التجارب - Comprehensive Analysis")
    print("═"*100)

    base_path = 'dataset/simulation/full_experiment_300_students'

    # التحقق من وجود المجلد
    if not os.path.exists(base_path):
        print(f"❌ المجلد غير موجود: {base_path}")
        print("   تأكد من تشغيل التجارب أولاً!")
        return

    # البحث عن جميع المجلدات الفرعية (كل مجلد يمثل تجربة واحدة)
    subfolders = [f for f in os.listdir(base_path)
                  if os.path.isdir(os.path.join(base_path, f)) and not f.startswith('.')]

    if not subfolders:
        print("❌ لم يتم العثور على أي مجلدات نتائج")
        print("   تأكد من اكتمال التجارب بنجاح!")
        return

    print(f"✅ تم العثور على {len(subfolders)} مجلد نتائج")
    print("─"*100)

    all_results = []

    # تحليل كل مجلد نتائج
    for folder_name in sorted(subfolders):
        # استخراج معلومات التجربة من اسم المجلد
        # مثال: real_only_recent_one_reflect-no_KM+PM+MM+CM_standard_cog_example-no_gpt-4o-mini
        parts = folder_name.split('_')
        if len(parts) < 2:
            continue

        experiment_type = parts[0]  # 'real' أو 'sim'
        model_name = parts[-1] if parts else 'unknown'

        # ترجمة اسم النموذج للعرض
        model_display_map = {
            'gpt-3.5-turbo': 'GPT-3.5 Turbo',
            'gpt-4o-mini': 'GPT-4o Mini',
            'llama-3.1-8b': 'Llama 3.1 8B',
            'qwen2.5-7b': 'Qwen2.5 7B (عربي)',
            'silma-9b': 'Phi-3 Medium'
        }
        model_display = model_display_map.get(model_name, model_name)

        folder_path = os.path.join(base_path, folder_name, 'log')

        if not os.path.exists(folder_path):
            continue

        # البحث عن ملفات النتائج
        dur_files = glob.glob(os.path.join(folder_path, '*_result_ind_dur.csv'))
        post_files = glob.glob(os.path.join(folder_path, '*_result_ind_post.csv'))

        if not dur_files:
            continue

        exp_label = 'التجربة 1: التنبؤ (Real)' if experiment_type == 'real' else 'التجربة 2: التوليد (Sim)'

        print(f"\n🤖 النموذج: {model_display}")
        print(f"📋 {exp_label}")
        print(f"👥 عدد الطلاب: {len(dur_files)}")

        # تجميع البيانات من جميع الطلاب
        all_gaze = []
        all_motor = []
        all_engagement = []
        all_confusion = []
        all_choice_sim = []
        all_answer_acc = []

        for dur_file in dur_files:
            try:
                df_dur = pd.read_csv(dur_file, encoding='utf-8-sig')

                if 'gaze_aoi_accuracy' in df_dur.columns:
                    all_gaze.append(df_dur['gaze_aoi_accuracy'].mean())

                if 'motor_aoi_accuracy' in df_dur.columns:
                    all_motor.append(df_dur['motor_aoi_accuracy'].mean())

                if 'engagement_accuracy' in df_dur.columns:
                    all_engagement.append(df_dur['engagement_accuracy'].mean())

                if 'confusion_accuracy' in df_dur.columns:
                    all_confusion.append(df_dur['confusion_accuracy'].mean())
            except:
                pass

        for post_file in post_files:
            try:
                df_post = pd.read_csv(post_file, encoding='utf-8-sig')

                if 'choice_similarity' in df_post.columns:
                    all_choice_sim.append(df_post['choice_similarity'].mean())

                if 'accuracy_similarity' in df_post.columns:
                    all_answer_acc.append(df_post['accuracy_similarity'].mean())
            except:
                pass

        # حساب المتوسطات
        result = {
            'model': model_display,
            'experiment': 'Real' if experiment_type == 'real' else 'Sim',
            'num_students': len(dur_files),
            'gaze_accuracy': np.nanmean(all_gaze) if all_gaze else 0,
            'motor_accuracy': np.nanmean(all_motor) if all_motor else 0,
            'engagement': np.nanmean(all_engagement) if all_engagement else 0,
            'confusion': np.nanmean(all_confusion) if all_confusion else 0,
            'choice_similarity': np.nanmean(all_choice_sim) if all_choice_sim else 0,
            'answer_accuracy': np.nanmean(all_answer_acc) if all_answer_acc else 0,
        }

        print(f"   👁️  دقة النظر: {result['gaze_accuracy']*100:.2f}%")
        print(f"   🖱️  دقة الماوس: {result['motor_accuracy']*100:.2f}%")
        print(f"   ✅  تطابق الإجابات: {result['choice_similarity']*100:.2f}%")
        print(f"   📚  دقة الإجابة العلمية: {result['answer_accuracy']*100:.2f}%")

        all_results.append(result)

    if not all_results:
        print("\n❌ لم يتم العثور على أي نتائج للتحليل")
        return

    df_results = pd.DataFrame(all_results)

    # ═══════════════════════════════════════════════════════════════════
    # 📊 جدول المقارنة الشامل
    # ═══════════════════════════════════════════════════════════════════
    print("\n" + "═"*100)
    print("📊 جدول المقارنة الشامل بين جميع النماذج")
    print("═"*100)

    comparison = df_results[['model', 'experiment', 'num_students', 'gaze_accuracy',
                             'motor_accuracy', 'choice_similarity', 'answer_accuracy']].copy()
    comparison['gaze_accuracy'] = (comparison['gaze_accuracy'] * 100).round(2)
    comparison['motor_accuracy'] = (comparison['motor_accuracy'] * 100).round(2)
    comparison['choice_similarity'] = (comparison['choice_similarity'] * 100).round(2)
    comparison['answer_accuracy'] = (comparison['answer_accuracy'] * 100).round(2)

    print(comparison.to_string(index=False))

    # ═══════════════════════════════════════════════════════════════════
    # 🏆 أفضل النماذج
    # ═══════════════════════════════════════════════════════════════════
    print("\n" + "═"*100)
    print("🏆 أفضل النماذج لكل مقياس")
    print("═"*100)

    metrics = [
        ('gaze_accuracy', 'دقة النظر 👁️'),
        ('motor_accuracy', 'دقة الماوس 🖱️'),
        ('choice_similarity', 'تطابق الإجابات ✅'),
        ('answer_accuracy', 'دقة الإجابة العلمية 📚')
    ]

    for metric, label in metrics:
        best_idx = df_results[metric].idxmax()
        best = df_results.loc[best_idx]
        print(f"\n🥇 {label}:")
        print(f"   • النموذج: {best['model']}")
        print(f"   • التجربة: {best['experiment']}")
        print(f"   • القيمة: {best[metric]*100:.2f}%")

    # ═══════════════════════════════════════════════════════════════════
    # 📊 مقارنة Real vs Sim لكل نموذج
    # ═══════════════════════════════════════════════════════════════════
    print("\n" + "═"*100)
    print("📊 مقارنة Real vs Sim لكل نموذج")
    print("═"*100)

    for model in df_results['model'].unique():
        model_data = df_results[df_results['model'] == model]
        real = model_data[model_data['experiment'] == 'Real']
        sim = model_data[model_data['experiment'] == 'Sim']

        if len(real) > 0 and len(sim) > 0:
            print(f"\n🤖 {model}:")
            print(f"   👁️  دقة النظر: Real={real['gaze_accuracy'].values[0]*100:.2f}% | Sim={sim['gaze_accuracy'].values[0]*100:.2f}%")
            print(f"   🖱️  دقة الماوس: Real={real['motor_accuracy'].values[0]*100:.2f}% | Sim={sim['motor_accuracy'].values[0]*100:.2f}%")
            print(f"   ✅  تطابق الإجابات: Real={real['choice_similarity'].values[0]*100:.2f}% | Sim={sim['choice_similarity'].values[0]*100:.2f}%")

    # ═══════════════════════════════════════════════════════════════════
    # 💾 حفظ النتائج
    # ═══════════════════════════════════════════════════════════════════
    output_file = os.path.join(base_path, 'comprehensive_analysis.csv')
    df_results.to_csv(output_file, index=False, encoding='utf-8-sig')
    print("\n" + "═"*100)
    print(f"💾 تم حفظ النتائج التفصيلية في:")
    print(f"   {output_file}")
    print("═"*100)

# ملاحظة: لتشغيل التحليل يدوياً بعد اكتمال جميع التجارب، قم بإلغاء التعليق عن السطر التالي:
# analyze_all_experiments()
